# Artificial Intelligence 601.464 &mdash; Project 4
## Decision Trees: from Entropy to Ensembles (medical screening edition)

### Before you begin...
00. We are using a Jupyter Notebook environment (tutorial: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html).
01. **Read the entire notebook before beginning your work.** It is long on purpose: it is a course, not a quiz.
02. Check the submission deadline on Gradescope.
03. You will need a **microphone** (built-in laptop mics are fine). Several answers in this project are spoken, not written.

### What this project is
By the end you will have built, from scratch, every idea behind one of the most widely used model families in medicine and industry:
entropy &rarr; information gain &rarr; the ID3 algorithm &rarr; honest evaluation on **253,680 real survey respondents** &rarr; the ways trees fail
(overfitting, fragmentation, instability, blindness to rare but important outcomes) &rarr; the fixes (pruning, thresholds, bagging, random forests,
boosting) &rarr; the industrial tools (scikit-learn, gradient boosting). The running problem is a **diabetes screening tool** built from the
CDC's Behavioral Risk Factor Surveillance System (BRFSS 2015) survey, downloaded from Kaggle.

Every part builds on the previous one: the `entropy` you write in Part 1 is inside the `information_gain` of Part 2, which is inside the `id3`
of Part 3, which you port to arrays in Part 5 and wrap in ensembles in Part 6. Do the parts in order.

### You will be made to run into problems. That is deliberate.
Eight times in this notebook a cell is designed to crash, return nonsense, or give a result that looks wrong. Each one is marked
<span style="color:#eb6834"><b>&#129512; Deliberate problem</b></span>. When you hit one, do not search for a fix first: **work out why it happened**.
The spoken questions that follow each problem ask you to explain the cause, and that explanation is what we grade.

### Voice answers instead of written answers
Wherever an older version of this project asked for a paragraph, this one asks you to **talk**. A recorder is built into the notebook
(see Part 0.3). We introduced this format to mitigate the use of AI tools to generate the entire homework in one shot: a spoken
explanation of *your* results is hard to outsource and easy to check. Nearly every question asks you to quote numbers from your own outputs. You may speak from notes. You may re-record as often as you like. You may not read someone (or something) else's script;
TAs listen for whether you engage with your own numbers.

### General directions for this assignment
00. Output format should be exactly as requested.
01. Functions should do only one thing.
02. Keep functions to about 20 lines or fewer (empty lines are fine, there is leeway, but do not blatantly ignore this). `lint_notebook_functions()` in Part 8 checks this for you.
03. Add a docstring to every function you write.
04. Do not modify the cells marked *course infrastructure* or the self-check cells. You may add cells for your own experiments.

### Grading at a glance (200 points)
| Component | Points | What TAs look at |
|---|---:|---|
| Code (self-checked functions and experiments) | 110 | The asserts in the notebook plus a few hidden tests with the same function signatures |
| Voice answers (20 recordings) | 70 | Correct reasoning, your own numbers, the cause of each deliberate problem |
| Process | 20 | Restart &amp; Run All, docstrings, function length, complete submission |

### Estimated time
| Part | Topic | Time |
|---|---|---|
| 0 | Setup, recorder, data download | 0.5 h |
| 1 | Entropy | 1.5 h |
| 2 | Information gain and its bias | 1.5 h |
| 3 | ID3 and prediction | 2.5 h |
| 4 | Real patients and honest metrics | 2.5 h |
| 5 | The limits of a single tree | 3 h |
| 6 | Ensembles | 3 h |
| 7 | Industry tools and the clinical decision | 2.5 h |
| 8 | Submit | 0.3 h |

### Before you submit...
00. Re-read the general directions above.
01. Hit "Kernel" &rarr; "Restart Kernel and Run All Cells". The first cell that runs should show [1], the second [2], and so on.
02. Run `build_submission()` at the end; it checks your 20 recordings and writes `submission.zip`.
03. Submit `submission.zip` (notebook + recordings) on Gradescope. Do not submit anything else.

## Part 0 &mdash; Setup

### 0.1 Environment
Create and activate a fresh environment, then install the requirements (a `requirements.txt` with the same list ships with this notebook):

```
conda create -n "ai_hw4" python=3.11 -y
conda activate "ai_hw4"
pip install numpy pandas matplotlib scikit-learn kagglehub anywidget ipywidgets jupyterlab
jupyter lab
```

Open this notebook in **JupyterLab or Jupyter Notebook in a normal browser** (Chrome, Edge, Firefox or Safari). The voice recorder uses the
browser's microphone API, which browsers only expose on `http://localhost` or `https://` addresses; the default Jupyter URL qualifies.
VS Code's notebook view may block the microphone, so use the browser for the recording cells even if you code elsewhere.
`xgboost` is optional (Part 7 detects whether it is installed).

**Imports.** Everything we need, in one place. Your own functions must use only the standard library and `numpy` until Part 7, where
scikit-learn appears.

In [ ]:
# =============================================================================
# Project 4: Decision Trees, from Entropy to Ensembles
# EN.601.464/664 Artificial Intelligence, Johns Hopkins University, Fall 2026
#
# Who worked on this
#   Musab Hadque <mhaque5@jhu.edu>, instructor of record for EN.601.464: course content, the lecture material
#       (the lecture dataset and tree that the self-checks reproduce) and approval of the design.
#
#   Matthew Brownrigg <mbrownrigg@jhu.edu>, teaching assistant: assignment design, review, testing and grading.
#
#   Kaleb Aklilu <kaklilu1@jh.edu>, teaching assistant: assignment design, requirements, grading policy, review.
#
#   Claude Code: drafted the refactored notebook, the
#       voice-recorder widget, the autograder and the TA rubric under the TAs' direction,
#       October 2026.
#
# Why voice answers: written explanations were replaced by spoken ones to mitigate the use
# of AI tools to generate the entire homework in one shot. Every spoken prompt asks for
# numbers taken from the student's own outputs, which is hard to outsource and easy to check.
#
# License: MIT (see LICENSE in the repository). Dataset: CDC BRFSS 2015 via Kaggle (CC0).
# Repository: https://github.com/kalk-ak/artificial_intelligence_decision_tree_hw
# =============================================================================

import math
import random
import time
import json
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", __import__("matplotlib").__version__)

### 0.2 Course infrastructure (do not edit)
The next cell defines the voice recorder, the submission builder, a small linter for your functions, the plotting style and the data loader.
You do not need to read it now, but you are welcome to: the recorder is about 80 lines of JavaScript talking to about 40 lines of Python through
a synced widget model, which is a nice example of how notebooks reach the browser.

In [ ]:
# =====================================================================================
#  COURSE INFRASTRUCTURE  --  please do not edit this cell.
#  Authorship and license: see the comment block at the top of the imports cell above (MIT).
#  It provides: the voice recorder, the question registry helpers, the submission
#  builder, a function linter, the plotting style and the Kaggle data loader.
# =====================================================================================
import base64
import datetime as _dt
import hashlib
import inspect
import json
import os
import pathlib
import textwrap
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    import anywidget
    import traitlets
    HAVE_WIDGET = True
except ImportError:
    HAVE_WIDGET = False

COURSE = "EN.601.464 Artificial Intelligence - Project 4: Decision Trees"
NOTEBOOK_FILENAME = "project_4_decision_trees.ipynb"
RECORDER_VERSION = "1.0"
RECORDINGS_DIR = pathlib.Path("recordings")
MANIFEST_PATH = RECORDINGS_DIR / "manifest.json"
MIME_EXT = {"audio/webm": "webm", "audio/mp4": "m4a", "audio/ogg": "ogg", "audio/wav": "wav", "audio/mpeg": "mp3"}
AUDIO_EXTS = {".webm", ".m4a", ".mp4", ".ogg", ".wav", ".mp3"}

# ----------------------------------------------------------------------------- voice recorder
VOICE_JS = r"""
function fmt(s) { s = Math.max(0, Math.floor(s)); return `${Math.floor(s / 60)}:${String(s % 60).padStart(2, "0")}`; }
function pickMime() {
  if (typeof MediaRecorder === "undefined") return null;
  const candidates = ["audio/webm;codecs=opus", "audio/webm", "audio/mp4", "audio/ogg;codecs=opus", ""];
  return candidates.find(m => m === "" || MediaRecorder.isTypeSupported(m));
}
function render({ model, el }) {
  const card = document.createElement("div");
  card.className = "vr-card";
  card.innerHTML = `
    <div class="vr-head"><span class="vr-badge"></span><span class="vr-title">Voice answer</span><span class="vr-target"></span></div>
    <div class="vr-prompt"></div>
    <div class="vr-controls">
      <button class="vr-btn vr-rec" type="button">&#9679; Record</button>
      <button class="vr-btn vr-stop" type="button" disabled>&#9632; Stop &amp; save</button>
      <span class="vr-timer">0:00</span>
    </div>
    <audio class="vr-audio" controls style="display:none"></audio>
    <div class="vr-status"></div>`;
  el.appendChild(card);
  const $ = (sel) => card.querySelector(sel);
  const qid = model.get("question_id"), minS = model.get("min_seconds"), maxS = model.get("max_seconds");
  $(".vr-badge").textContent = "\u{1F399} " + qid;
  $(".vr-prompt").textContent = model.get("prompt_text");
  $(".vr-target").textContent = `aim for ${fmt(minS)}–${fmt(maxS)}`;
  const recBtn = $(".vr-rec"), stopBtn = $(".vr-stop"), timerEl = $(".vr-timer"), audio = $(".vr-audio"), statusEl = $(".vr-status");
  const setStatus = (msg, kind) => { statusEl.textContent = msg || ""; statusEl.dataset.kind = kind || ""; };
  const existing = model.get("existing_b64");
  if (existing) { audio.src = `data:${model.get("existing_mime")};base64,${existing}`; audio.style.display = "block"; recBtn.innerHTML = "&#9679; Re-record"; }
  if (model.get("status")) setStatus(model.get("status"), model.get("status_kind"));
  let recorder = null, chunks = [], stream = null, timer = null, started = 0;
  async function start() {
    if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
      setStatus("Microphone API unavailable here. Open the notebook in Chrome, Edge, Firefox or Safari at http://localhost or an https address.", "err");
      return;
    }
    const mime = pickMime();
    if (mime === null) { setStatus("This browser has no MediaRecorder support. Try Chrome, Edge or Firefox.", "err"); return; }
    try { stream = await navigator.mediaDevices.getUserMedia({ audio: true }); }
    catch (e) { setStatus("Microphone permission denied or no microphone found: " + e.message, "err"); return; }
    chunks = [];
    recorder = mime ? new MediaRecorder(stream, { mimeType: mime }) : new MediaRecorder(stream);
    recorder.ondataavailable = (e) => { if (e.data && e.data.size > 0) chunks.push(e.data); };
    recorder.onstop = finish;
    recorder.start(250);
    started = Date.now();
    recBtn.disabled = true; stopBtn.disabled = false; card.classList.add("vr-recording");
    setStatus("Recording… speak normally. Click Stop & save when you are done.", "rec");
    timer = setInterval(() => {
      const s = (Date.now() - started) / 1000;
      timerEl.textContent = fmt(s);
      if (s >= maxS) stop();
    }, 200);
  }
  function stop() { if (recorder && recorder.state !== "inactive") recorder.stop(); }
  function finish() {
    clearInterval(timer);
    const seconds = (Date.now() - started) / 1000;
    if (stream) stream.getTracks().forEach(t => t.stop());
    const type = recorder.mimeType || (chunks[0] && chunks[0].type) || "audio/webm";
    const blob = new Blob(chunks, { type });
    audio.src = URL.createObjectURL(blob); audio.style.display = "block";
    recBtn.disabled = false; stopBtn.disabled = true; card.classList.remove("vr-recording");
    recBtn.innerHTML = "&#9679; Re-record";
    setStatus("Saving…", "");
    const reader = new FileReader();
    reader.onload = () => {
      const b64 = String(reader.result).split(",")[1] || "";
      model.set("mime_type", type);
      model.set("duration", Math.round(seconds * 10) / 10);
      model.set("audio_b64", b64);
      model.save_changes();
    };
    reader.readAsDataURL(blob);
  }
  recBtn.addEventListener("click", start);
  stopBtn.addEventListener("click", stop);
  model.on("change:status", () => setStatus(model.get("status"), model.get("status_kind")));
}
export default { render };
"""

VOICE_CSS = """
.vr-card { font-family: system-ui, -apple-system, "Segoe UI", sans-serif; border: 1px solid var(--jp-border-color2, #cfcfcf);
  border-left: 6px solid #2a78d6; border-radius: 8px; padding: 12px 16px; margin: 6px 0; max-width: 820px;
  background: var(--jp-layout-color1, #fff); color: var(--jp-ui-font-color1, #111); }
.vr-card.vr-recording { border-left-color: #d03b3b; }
.vr-head { display: flex; align-items: baseline; gap: 10px; margin-bottom: 6px; }
.vr-badge { font-weight: 700; font-size: 1.05em; }
.vr-title { color: var(--jp-ui-font-color2, #555); }
.vr-target { margin-left: auto; font-size: 0.85em; color: var(--jp-ui-font-color2, #666); }
.vr-prompt { white-space: pre-wrap; line-height: 1.45; margin: 4px 0 10px 0; }
.vr-controls { display: flex; align-items: center; gap: 10px; margin-bottom: 8px; }
.vr-btn { border: 1px solid var(--jp-border-color1, #999); background: var(--jp-layout-color2, #f3f3f3); color: inherit;
  border-radius: 6px; padding: 6px 14px; font-size: 0.95em; cursor: pointer; }
.vr-btn:disabled { opacity: 0.45; cursor: default; }
.vr-rec { color: #b11; font-weight: 600; }
.vr-timer { font-variant-numeric: tabular-nums; font-weight: 600; min-width: 3em; }
.vr-audio { width: 100%; max-width: 480px; margin-bottom: 6px; }
.vr-status { font-size: 0.9em; min-height: 1.2em; color: var(--jp-ui-font-color2, #555); }
.vr-status[data-kind="ok"] { color: #0a7a0a; }
.vr-status[data-kind="warn"] { color: #a85d00; }
.vr-status[data-kind="err"] { color: #c21d1d; }
.vr-status[data-kind="rec"] { color: #c21d1d; font-weight: 600; }
"""


def load_manifest():
    """Read recordings/manifest.json, returning an empty manifest if none exists yet."""
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"version": RECORDER_VERSION, "course": COURSE, "recordings": {}}


def save_manifest(manifest):
    """Write the manifest to disk (pretty-printed so that TAs can read it)."""
    RECORDINGS_DIR.mkdir(exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))


def manifest_entry(path, raw_bytes, mime, duration, prompt_text, external=False):
    """Build the manifest record for one recording."""
    return {"file": path.name, "mime": mime, "duration_s": float(duration), "bytes": len(raw_bytes),
            "sha256": hashlib.sha256(raw_bytes).hexdigest(),
            "recorded_at": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"),
            "prompt": prompt_text, "recorder_version": RECORDER_VERSION, "external": external}


def store_recording(question_id, raw_bytes, mime, duration, prompt_text="", external=False):
    """Save one recording as recordings/<question_id>.<ext>, replacing older takes, and log it in the manifest."""
    RECORDINGS_DIR.mkdir(exist_ok=True)
    ext = MIME_EXT.get(mime.split(";")[0].strip().lower(), "webm")
    for stale in RECORDINGS_DIR.glob(f"{question_id}.*"):
        stale.unlink()
    path = RECORDINGS_DIR / f"{question_id}.{ext}"
    path.write_bytes(raw_bytes)
    manifest = load_manifest()
    manifest["recordings"][question_id] = manifest_entry(path, raw_bytes, mime, duration, prompt_text, external)
    save_manifest(manifest)
    return path


if HAVE_WIDGET:
    class VoiceRecorder(anywidget.AnyWidget):
        """Record/stop widget: the browser captures audio, Python saves the bytes the moment they arrive."""
        _esm = VOICE_JS
        _css = VOICE_CSS
        question_id = traitlets.Unicode("").tag(sync=True)
        prompt_text = traitlets.Unicode("").tag(sync=True)
        min_seconds = traitlets.Int(30).tag(sync=True)
        max_seconds = traitlets.Int(120).tag(sync=True)
        audio_b64 = traitlets.Unicode("").tag(sync=True)
        mime_type = traitlets.Unicode("").tag(sync=True)
        duration = traitlets.Float(0.0).tag(sync=True)
        status = traitlets.Unicode("").tag(sync=True)
        status_kind = traitlets.Unicode("").tag(sync=True)
        existing_b64 = traitlets.Unicode("").tag(sync=True)
        existing_mime = traitlets.Unicode("").tag(sync=True)

        @traitlets.observe("audio_b64")
        def _on_audio(self, change):
            """Runs automatically when the browser sends a finished recording."""
            if not change["new"]:
                return
            try:
                path = store_recording(self.question_id, base64.b64decode(change["new"]), self.mime_type, self.duration, self.prompt_text)
                short = self.duration < self.min_seconds
                self.status_kind = "warn" if short else "ok"
                tail = "That is shorter than the target; consider recording again." if short else "You may re-record; the newest take replaces the old one."
                self.status = f"Saved {path} ({self.duration:.0f} s). {tail}"
            except Exception as exc:
                self.status_kind, self.status = "err", f"Could not save the recording: {exc!r}"
            finally:
                self.audio_b64 = ""


def existing_recording_kwargs(question_id):
    """Widget arguments that replay the previous take of a question, if one is on disk."""
    entry = load_manifest()["recordings"].get(question_id)
    if not entry or not (RECORDINGS_DIR / entry["file"]).exists():
        return {}
    data = (RECORDINGS_DIR / entry["file"]).read_bytes()
    note = f"Existing recording from {entry['recorded_at']} ({entry['duration_s']:.0f} s). Record again to replace it."
    if len(data) > 8_000_000:
        return {"status": note, "status_kind": "ok"}
    return {"existing_b64": base64.b64encode(data).decode(), "existing_mime": entry["mime"], "status": note, "status_kind": "ok"}


def record_answer(question_id):
    """Show the recorder for one voice question (looked up in VOICE_QUESTIONS). Recording saves itself."""
    q = VOICE_QUESTIONS[question_id]
    if not HAVE_WIDGET:
        print(f"[{question_id}] anywidget is not installed, so the in-notebook recorder cannot run.\n"
              "Run `pip install anywidget ipywidgets`, restart the kernel and re-run this cell.\n"
              "Fallback: record with any app and call register_external_recording(question_id, 'path/to/file.m4a', seconds).\n"
              f"Question: {q['prompt']}")
        return
    display(VoiceRecorder(question_id=question_id, prompt_text=q["prompt"], min_seconds=q["min_s"], max_seconds=q["max_s"],
                          **existing_recording_kwargs(question_id)))


def register_external_recording(question_id, file_path, duration_s):
    """Fallback if the browser recorder cannot run: attach an audio file you recorded elsewhere (flagged for TAs)."""
    p = pathlib.Path(file_path)
    if p.suffix.lower() not in AUDIO_EXTS:
        raise ValueError(f"Unsupported audio extension {p.suffix}; use one of {sorted(AUDIO_EXTS)}")
    mime = next((m for m, e in MIME_EXT.items() if e == p.suffix.lower().lstrip(".")), "audio/mp4")
    path = store_recording(question_id, p.read_bytes(), mime, duration_s, VOICE_QUESTIONS[question_id]["prompt"], external=True)
    print(f"Registered {path} ({duration_s:.0f} s) as an external recording for {question_id}.")


def recording_status():
    """Table of every voice question: recorded or not, duration, when, and whether it meets the target length."""
    manifest = load_manifest()["recordings"]
    rows = []
    for qid, q in VOICE_QUESTIONS.items():
        e = manifest.get(qid)
        ok = bool(e) and (RECORDINGS_DIR / e["file"]).exists()
        rows.append({"question": qid, "part": q["part"], "recorded": "yes" if ok else "MISSING",
                     "duration_s": round(e["duration_s"]) if ok else None, "target": f"{q['min_s']}-{q['max_s']} s",
                     "length_ok": (e["duration_s"] >= q["min_s"]) if ok else False,
                     "recorded_at": e["recorded_at"][:16].replace("T", " ") if ok else "", "external": e.get("external", False) if ok else ""})
    return pd.DataFrame(rows).set_index("question")


def verify_recordings():
    """Return (problems, warnings) lists after checking files, hashes and durations against the manifest."""
    manifest = load_manifest()["recordings"]
    problems, warnings = [], []
    for qid, q in VOICE_QUESTIONS.items():
        e = manifest.get(qid)
        if not e or not (RECORDINGS_DIR / e["file"]).exists():
            problems.append(f"{qid}: no recording found (run its record_answer cell).")
            continue
        data = (RECORDINGS_DIR / e["file"]).read_bytes()
        if hashlib.sha256(data).hexdigest() != e["sha256"]:
            problems.append(f"{qid}: file changed after it was recorded (hash mismatch).")
        if e["duration_s"] < q["min_s"]:
            warnings.append(f"{qid}: {e['duration_s']:.0f} s is shorter than the {q['min_s']} s target.")
        if e.get("external"):
            warnings.append(f"{qid}: recorded outside the notebook (allowed, flagged for TAs).")
    return problems, warnings


def write_review_page():
    """Create recordings/review.html: one row per question with an inline player, for TAs."""
    manifest = load_manifest()["recordings"]
    rows = []
    for qid, q in VOICE_QUESTIONS.items():
        e = manifest.get(qid)
        player = f'<audio controls preload="none" src="{e["file"]}"></audio>' if e else "<em>missing</em>"
        meta = f'{e["duration_s"]:.0f} s &middot; {e["recorded_at"][:16]}' + (" &middot; EXTERNAL" if e and e.get("external") else "") if e else ""
        rows.append(f"<tr><td><b>{qid}</b><br><small>Part {q['part']}</small></td><td>{q['prompt']}</td><td>{player}<br><small>{meta}</small></td></tr>")
    html = ("<!doctype html><html><head><meta charset='utf-8'><title>Voice answers</title><style>body{font-family:system-ui,sans-serif;margin:24px}"
            "table{border-collapse:collapse;width:100%}td{border-top:1px solid #ddd;padding:10px;vertical-align:top}td:nth-child(2){width:55%}</style></head>"
            f"<body><h2>{COURSE} &mdash; voice answers</h2><table>{''.join(rows)}</table></body></html>")
    (RECORDINGS_DIR / "review.html").write_text(html)


def build_submission(notebook_filename=NOTEBOOK_FILENAME):
    """Check the notebook and all recordings, then write submission.zip for Gradescope."""
    problems, warnings = verify_recordings()
    nb = pathlib.Path(notebook_filename)
    if not nb.exists():
        problems.append(f"Notebook {notebook_filename} not found in {os.getcwd()} (pass notebook_filename=... if you renamed it).")
    for w in warnings:
        print("WARNING:", w)
    if problems:
        missing = [p.split(":")[0] for p in problems if "no recording found" in p]
        others = [p for p in problems if "no recording found" not in p]
        print("Submission NOT built. Fix these first:")
        if missing:
            print(f"  - {len(missing)} recording(s) missing: {', '.join(missing)}  (run their record_answer cells)")
        for p in others:
            print("  -", p)
        return None
    write_review_page()
    out = pathlib.Path("submission.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(nb, nb.name)
        for f in sorted(RECORDINGS_DIR.iterdir()):
            z.write(f, f"recordings/{f.name}")
    saved = _dt.datetime.fromtimestamp(nb.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    print(f"Built {out} ({out.stat().st_size / 1e6:.1f} MB) with {nb.name} (last saved {saved}) and {len(VOICE_QUESTIONS)} recordings.")
    print("If that 'last saved' time is older than your latest edits, save the notebook and run this cell again.")
    return out


# ----------------------------------------------------------------------------- function linter
def function_body_length(fn):
    """Number of code lines in a function's body, not counting blank lines, comments and the docstring."""
    import ast
    src = textwrap.dedent(inspect.getsource(fn))
    node = ast.parse(src).body[0]
    body = node.body
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) and isinstance(body[0].value.value, str):
        body = body[1:]
    if not body:
        return 0
    lines = src.splitlines()[body[0].lineno - 1: body[-1].end_lineno]
    return sum(1 for line in lines if line.strip() and not line.strip().startswith("#"))


def lint_notebook_functions(max_lines=20, namespace=None):
    """List the functions YOU defined in this notebook with their docstring status and body length."""
    ns = namespace if namespace is not None else globals()
    rows = []
    for name, obj in list(ns.items()):
        if inspect.isfunction(obj) and obj.__module__ == "__main__" and name not in _INFRA_NAMES:
            try:
                n = function_body_length(obj)
            except (OSError, TypeError):
                n = None
            rows.append({"function": name, "has_docstring": bool(obj.__doc__), "body_lines": n,
                         "ok": bool(obj.__doc__) and (n is not None and n <= max_lines)})
    table = pd.DataFrame(rows).sort_values(["ok", "function"]).reset_index(drop=True)
    flagged = table.loc[~table["ok"], "function"].tolist()
    if flagged:
        print(f"{len(flagged)} function(s) need attention (missing docstring or longer than {max_lines} lines): {', '.join(flagged)}")
    else:
        print(f"All {len(table)} functions have docstrings and are within the {max_lines}-line guideline.")
    with pd.option_context("display.max_rows", 200):
        display(table)
    return None


# ----------------------------------------------------------------------------- plotting style
PALETTE = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100",
           "magenta": "#e87ba4", "green": "#008300", "violet": "#6250d6", "red": "#e34948", "gray": "#898781"}
SERIES = [PALETTE["blue"], PALETTE["orange"], PALETTE["aqua"], PALETTE["yellow"]]
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7.2, 3.8), "font.family": "sans-serif", "font.size": 10,
                     "lines.linewidth": 1.8, "lines.markersize": 6, "axes.titlelocation": "left", "axes.titlesize": 11,
                     "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False})


def style_axes(ax, title=None, xlabel=None, ylabel=None):
    """Quiet chart chrome: hairline grid, no top/right spines, left-aligned title."""
    ax.spines[["top", "right"]].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.grid(True, axis="y", color="#e1e0d9", linewidth=1)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, color="#0b0b0b")
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    return ax


# ----------------------------------------------------------------------------- data loader
DATASET_HANDLE = "alexteboul/diabetes-health-indicators-dataset"
DATA_FILE = "diabetes_binary_health_indicators_BRFSS2015.csv"


def load_diabetes_data():
    """Download the CDC BRFSS 2015 diabetes indicators from Kaggle (cached) and return them as an in-memory DataFrame."""
    import kagglehub
    try:
        folder = kagglehub.dataset_download(DATASET_HANDLE)
    except Exception as exc:
        raise RuntimeError("Kaggle download failed. Check your internet connection; if Kaggle asks for credentials, run "
                           "`import kagglehub; kagglehub.login()` in a cell and re-run.") from exc
    df = pd.read_csv(os.path.join(folder, DATA_FILE))
    return df.astype("int64")


_INFRA_NAMES = set(globals())
print("Course infrastructure loaded. Voice recorder available:", HAVE_WIDGET)

In [ ]:
# Every voice question in this project, in one place (the recorder and the submission checker read this).
VOICE_QUESTIONS = {
    "Q00": {"part": 0, "min_s": 20, "max_s": 60,
             "prompt": ("Mic check. Say your full name and your JHED ID. Then, without looking anything up, describe in one or "
                        "two sentences what you think a decision tree does. There is no wrong answer; you will compare it with "
                        "your final answer at the end.")},
    "Q01": {"part": 1, "min_s": 45, "max_s": 90,
             "prompt": ("In your own words: why does a group that is half diabetic and half not have an entropy of exactly 1 bit, "
                        "and why does a group where everyone has the same label have 0 bits? What is a 'bit' measuring here? "
                        "Finish by saying the entropy you computed for the real diabetes label column and what that number tells "
                        "you about the problem.")},
    "Q02": {"part": 1, "min_s": 30, "max_s": 75,
             "prompt": ("Describe what happened when you first ran entropy_from_probabilities on the pure group [1.0, 0.0]. Why "
                        "does the formula try to compute log(0)? What is the limit of p times log2(p) as p goes to 0, and what "
                        "exactly did you change in your code so that a pure group returns 0?")},
    "Q03": {"part": 2, "min_s": 60, "max_s": 120,
             "prompt": ("Walk me through the information gain of BMI_class on the toy patients: the entropy of the parent, the "
                        "entropy of each of the three child groups, the weight each child gets, the remainder, and the final "
                        "gain. Say the numbers you got. Then explain why each child's entropy is weighted by its size instead of "
                        "being averaged equally.")},
    "Q04": {"part": 2, "min_s": 60, "max_s": 120,
             "prompt": ("When you added patient_id, information gain chose it as the best attribute. Explain why its gain equals "
                        "the parent's entropy, and what would happen if the tree used that split on the next patient who walks "
                        "in. What does gain ratio divide by, which attribute did it prefer in your run, and by how much? Does "
                        "gain ratio fully fix the problem?")},
    "Q05": {"part": 3, "min_s": 60, "max_s": 120,
             "prompt": ("List every condition under which your id3 function stops splitting and returns a leaf, and say what "
                        "label that leaf gets in each case. In particular: what do you return for an empty branch, and why is the "
                        "parent's majority the right choice?")},
    "Q06": {"part": 3, "min_s": 45, "max_s": 90,
             "prompt": ("Your naive predictor returned None for the underweight patient. Explain exactly where the tree walk got "
                        "stuck and why. Describe your fix. Why is the majority class stored at that node a sensible fallback, and "
                        "what would be a worse alternative?")},
    "Q07": {"part": 3, "min_s": 30, "max_s": 60,
             "prompt": ("HighBP had the second-highest information gain at the root of the toy data, yet it does not appear "
                        "anywhere in the final tree. Explain how a greedy, recursive algorithm can end up never using a globally "
                        "useful feature.")},
    "Q08": {"part": 4, "min_s": 60, "max_s": 120,
             "prompt": ("Your trees from depth 0 upward all scored about 86% on the real patients, and deeper trees were not "
                        "better. Explain why, using the prevalence of diabetes in the data and what a leaf needs to contain "
                        "before its prediction changes. Say the sensitivity you measured at depth 0 and at the deepest tree. Is "
                        "an 86%-accurate screening tool useful?")},
    "Q09": {"part": 4, "min_s": 60, "max_s": 120,
             "prompt": ("Define sensitivity and specificity in terms of the four cells of your confusion matrix, and state your "
                        "numbers for one of your trees. If this tree were used as a first-pass diabetes screen in a clinic, which "
                        "of the two matters more and why? Which would matter more for a confirmatory test before starting "
                        "medication?")},
    "Q10": {"part": 4, "min_s": 30, "max_s": 75,
             "prompt": ("About 14% of the rows in this dataset have an exact duplicate. If a patient's row appears in both the "
                        "training fold and the test fold, what happens to the estimated test performance, and is that leakage? "
                        "What would you do about it before trusting a cross-validation number?")},
    "Q11": {"part": 5, "min_s": 60, "max_s": 120,
             "prompt": ("Describe the shape of your training-error and test-error curves as the depth limit grows. At what depth "
                        "did test error reach its minimum in your run? Why does training error keep falling while test error "
                        "turns around, and what is the name for what happens on the right side of the plot?")},
    "Q12": {"part": 5, "min_s": 45, "max_s": 90,
             "prompt": ("Compare the number of leaves and the unseen-branch rate of your categorical ID3 tree against the "
                        "threshold tree, quoting your numbers. Why does splitting BMI into dozens of branches (one per integer "
                        "value) hurt generalisation even though it raises training accuracy? How does a threshold split avoid the "
                        "problem?")},
    "Q13": {"part": 5, "min_s": 60, "max_s": 120,
             "prompt": ("Trees A and B were trained on two halves of the same data. Compare their root splits and state the "
                        "disagreement rate between their predictions on the test set. Why are decision trees high-variance "
                        "models? Why should a physician be uneasy about a tool whose rules change this much when it is retrained?")},
    "Q14": {"part": 5, "min_s": 45, "max_s": 90,
             "prompt": ("Your tree uses Income and Education to split patients. Give one argument for allowing a clinical "
                        "screening tool to use such features and one argument against. Say which side you land on and what you "
                        "would want to check before deploying it.")},
    "Q15": {"part": 6, "min_s": 60, "max_s": 120,
             "prompt": ("Explain bootstrap aggregation in your own words. In your run, what did averaging 25 trees do to "
                        "accuracy, sensitivity and F1 compared with the single tree at the default 0.5 vote threshold? What "
                        "happened when you lowered the threshold, and why does the ensemble's vote fraction give you that knob "
                        "when a single tree did not?")},
    "Q16": {"part": 6, "min_s": 45, "max_s": 90,
             "prompt": ("Count how many of your 25 bagged trees shared the same root feature. Why are bootstrapped trees so "
                        "correlated, and why does correlation limit how much averaging can help? What does max_features change at "
                        "each node, and how did the root features and the metrics change in your random-forest run?")},
    "Q17": {"part": 6, "min_s": 60, "max_s": 120,
             "prompt": ("In AdaBoost, what happens to the weight of a misclassified patient after each round, and to a correctly "
                        "classified one? The first stump's weighted error was about 0.14 in your run; explain why that gives a "
                        "large alpha. Explain how the reweighting makes boosting pay attention to the diabetic minority that a "
                        "single tree ignored.")},
    "Q18": {"part": 7, "min_s": 45, "max_s": 90,
             "prompt": ("You trained on a balanced 50/50 sample and got sensitivity near 0.8, but precision collapsed when you "
                        "evaluated at the true 14% prevalence, even though precision looked fine on a balanced test set. Explain "
                        "this with base rates: out of 1,000 screened patients, roughly how many true positives and how many false "
                        "positives does your model produce?")},
    "Q19": {"part": 7, "min_s": 120, "max_s": 180,
             "prompt": ("Capstone. You are presenting to a Hopkins clinical team deciding whether to deploy a diabetes screening "
                        "model built from these survey indicators. Which model and operating threshold would you recommend, "
                        "quoting its sensitivity, specificity and precision from your results? Describe two ways it could fail in "
                        "practice, one ethical or practical concern, and what you would monitor after deployment. Finally, in one "
                        "sentence, how has your understanding of decision trees changed since your mic-check answer?")},
}
print(f"{len(VOICE_QUESTIONS)} voice questions registered; total target length "
      f"{sum(q['min_s'] for q in VOICE_QUESTIONS.values())//60}-{sum(q['max_s'] for q in VOICE_QUESTIONS.values())//60} minutes.")

### 0.3 Voice answers: how they work
Each voice question looks like the purple box below. Running the cell under it shows a recorder:

1. Click **Record**. The first time, your browser asks for microphone permission; allow it.
2. Speak. A timer runs; each question states a target length. The recorder stops by itself at the maximum.
3. Click **Stop &amp; save**. The audio is sent to Python and written to `recordings/<question id>.webm` (or `.m4a` on Safari),
   together with its length, a timestamp and a fingerprint in `recordings/manifest.json`. You will see a green confirmation line.
4. Press play on the audio bar to listen back. Record again if you like; only the newest take is kept.

If you re-run a cell (for example after *Restart &amp; Run All*) your previous take is shown, so re-running never erases a recording.
`recording_status()` (Part 8) lists what you have recorded so far. If the browser recorder does not work on your machine, record with any
app and attach the file with `register_external_recording("Q03", "path/to/file.m4a", seconds)`; TAs are told it was recorded outside the notebook.

**Why voice.** Written answers can be generated wholesale by an AI tool; a recording of you explaining the numbers in your own notebook
cannot, and that is the point. The rubric rewards engagement with your results, not polish.

**Rules.** Speak in your own words and your own voice. Notes are fine. Quote the numbers from your outputs when the question asks for them.
Recordings are only heard by course staff. Length guidance is guidance: a precise 40-second answer beats a rambling 3-minute one.

Start with the mic check.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q00 &nbsp;&middot;&nbsp; aim for 20&ndash;60 seconds</strong>

Mic check. Say your full name and your JHED ID. Then, without looking anything up, describe in one or two sentences what you think a decision tree does. There is no wrong answer; you will compare it with your final answer at the end.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q00")

### 0.4 The data: 253,680 patients, in memory
The CDC's **Behavioral Risk Factor Surveillance System** is the largest continuously conducted health survey in the world. The Kaggle dataset
[`alexteboul/diabetes-health-indicators-dataset`](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset) is a cleaned
extract of the 2015 wave: one row per respondent, 21 health indicators, and a label that says whether the respondent reported having
(pre)diabetes. Two honest caveats that will matter later: the answers are **self-reported**, and what we build is a **screening / risk
stratification** tool, not a diagnosis.

The cell below downloads the file through `kagglehub` (6 MB, public, no Kaggle account needed), caches it in your user cache folder, and loads it
**once into memory** as a pandas DataFrame. From here on everything works on that in-memory copy; nothing is written back to disk and nothing
about the data goes into your submission. If your network blocks Kaggle, the error message tells you what to do.

| Column | Meaning | Values |
|---|---|---|
| `Diabetes_binary` | **target**: no diabetes (0) vs prediabetes or diabetes (1) | 0/1 |
| `HighBP`, `HighChol` | ever told you have high blood pressure / cholesterol | 0/1 |
| `CholCheck` | cholesterol check in the last 5 years | 0/1 |
| `BMI` | body mass index | 12&ndash;98 |
| `Smoker` | smoked at least 100 cigarettes in your life | 0/1 |
| `Stroke`, `HeartDiseaseorAttack` | ever had a stroke / coronary heart disease or MI | 0/1 |
| `PhysActivity` | physical activity in the past 30 days | 0/1 |
| `Fruits`, `Veggies` | eat fruit / vegetables at least once a day | 0/1 |
| `HvyAlcoholConsump` | heavy drinker | 0/1 |
| `AnyHealthcare`, `NoDocbcCost` | any health coverage / could not see a doctor because of cost | 0/1 |
| `GenHlth` | self-rated general health, 1 excellent &hellip; 5 poor | 1&ndash;5 |
| `MentHlth`, `PhysHlth` | days of poor mental / physical health in the past 30 | 0&ndash;30 |
| `DiffWalk` | serious difficulty walking or climbing stairs | 0/1 |
| `Sex` | 0 female, 1 male | 0/1 |
| `Age` | 13 age bands, 1 = 18&ndash;24 &hellip; 13 = 80+ | 1&ndash;13 |
| `Education` | 1 never attended &hellip; 6 college graduate | 1&ndash;6 |
| `Income` | 1 under $10k &hellip; 8 over $75k | 1&ndash;8 |

In [ ]:
diabetes_df = load_diabetes_data()
print(f"{diabetes_df.shape[0]:,} respondents x {diabetes_df.shape[1]} columns; "
      f"{diabetes_df.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory")
print(f"Prevalence of (pre)diabetes in the data: {diabetes_df['Diabetes_binary'].mean():.4f}")
diabetes_df.head()

One number to remember: **13.93%** of respondents have the condition. Keep it in mind; it comes back in Part 4 with a vengeance.

## Part 1 &mdash; Uncertainty and entropy

### 1.1 Why we need a number for "how mixed is this group?"
A triage nurse with a waiting room of 100 patients, 14 of whom will turn out to have diabetes, can ask one question before deciding who gets a
blood test. "Do you have high blood pressure?" "Is your BMI over 30?" "Are you over 60?" Which question first? The intuitive answer is: the one
whose answer **sorts patients into groups that are as unmixed as possible**. To compare questions we need a number that says how mixed a
group is. That number is **entropy**, and the whole decision-tree algorithm is built on it.

For a group whose labels occur with probabilities $p_1, \dots, p_k$ (the fraction of each class), Shannon's entropy is

$$H = -\sum_{i=1}^{k} p_i \log_2 p_i \quad\text{(in bits)}$$

Intuition for the bits: $-\log_2 p_i$ is the number of yes/no questions you would need to pin down an outcome that had probability $p_i$,
and $H$ is the average of that over the outcomes. A fair coin takes exactly one question (1 bit). A group that is 100% one class needs no
questions (0 bits). Four equally likely classes need two (2 bits). A group that is 86% / 14% sits in between: it is *mostly* predictable, so
it carries less than a bit of uncertainty.

We build entropy in two layers, because the two layers fail in different ways and we want you to meet both failures:

* `entropy_from_probabilities(probs)`: the formula itself, given a list of probabilities.
* `entropy(labels)`: count the labels in a group, turn the counts into probabilities, call the first function.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>entropy_from_probabilities</code></strong>

Compute $H$ for a list of probabilities that sum to 1. Use <code>math.log2</code>. Write it the obvious way first; the next cells will tell you if the obvious way has a hole in it.
</div>

In [ ]:
def entropy_from_probabilities(probs: List[float]) -> float:
    """Shannon entropy in bits of a distribution given as a list of probabilities that sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError()

**Self-check.** Three groups you can verify by hand.

In [ ]:
assert math.isclose(entropy_from_probabilities([0.5, 0.5]), 1.0), "a fair coin is exactly 1 bit"
assert math.isclose(entropy_from_probabilities([0.25, 0.25, 0.25, 0.25]), 2.0), "four equally likely classes are 2 bits"
assert math.isclose(entropy_from_probabilities([0.86, 0.14]), 0.5842, abs_tol=1e-3), "an 86/14 group carries about 0.58 bits"
print("entropy_from_probabilities: basic checks passed")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #1: the pure group</strong>

A waiting room where <i>nobody</i> has diabetes has probabilities <code>[1.0, 0.0]</code>. Its entropy should be 0: no uncertainty at all. Run the cell below. If your function was written the obvious way it will stop with <code>ValueError: math domain error</code>. Do not fix it yet. First read on.
</div>

In [ ]:
print("entropy of a pure group:", entropy_from_probabilities([1.0, 0.0]))

### 1.2 Why it crashed, and why the fix is legitimate
The formula asks for $0 \cdot \log_2 0$, and $\log_2 0$ is undefined (it heads to $-\infty$). Mathematically the term is rescued by a limit:

$$\lim_{p \to 0^{+}} p \log_2 p = 0$$

($p$ goes to zero faster than $\log p$ goes to minus infinity), so by convention **$0 \log 0 = 0$** in information theory. A class that never
occurs contributes nothing to the uncertainty, which is exactly what intuition says.

**Fix your function** so that zero-probability classes are skipped (or contribute 0), re-run its cell, then run the check below. The crash was
not a Python quirk; it is the first place where the mathematics of trees needs a convention, and you will meet the consequence again when a
split produces an empty branch in Part 3.

In [ ]:
assert entropy_from_probabilities([1.0, 0.0]) == 0.0, "a pure group has zero entropy"
assert entropy_from_probabilities([0.0, 1.0, 0.0]) == 0.0
print("entropy_from_probabilities: pure-group check passed")

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>class_probabilities</code> and <code>entropy</code></strong>

<code>class_probabilities(labels)</code> returns the fraction of each distinct label as a dictionary (use <code>collections.Counter</code>); <code>entropy(labels)</code> turns a list of labels into probabilities and calls <code>entropy_from_probabilities</code>. An empty list has entropy 0 (there is nothing to be uncertain about), and dividing by zero is not allowed to happen.
</div>

In [ ]:
def class_probabilities(labels: List) -> Dict:
    """Map each distinct label to the fraction of the list it occupies, e.g. ['no','no','yes'] -> {'no': 2/3, 'yes': 1/3}."""
    # YOUR CODE HERE
    raise NotImplementedError()


def entropy(labels: List) -> float:
    """Entropy in bits of a list of class labels (0.0 for an empty list)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert class_probabilities(['no', 'no', 'yes', 'no']) == {'no': 0.75, 'yes': 0.25}
assert entropy([]) == 0.0, "empty group"
assert entropy(['yes'] * 7) == 0.0, "pure group"
assert math.isclose(entropy(['yes', 'no']), 1.0)
assert math.isclose(entropy(['no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'yes', 'no', 'no']), 0.9968, abs_tol=1e-3), "the lecture data has 8 no / 7 yes"
print("entropy: checks passed")

### 1.3 The entropy of the real problem
Now the first contact between your code and the 253,680 patients: the entropy of the diabetes label itself. This is the total uncertainty a
perfect screening tool would have to remove.

In [ ]:
real_labels = diabetes_df["Diabetes_binary"].tolist()
H_diabetes = entropy(real_labels)
print(f"Entropy of the diabetes label over {len(real_labels):,} respondents: {H_diabetes:.4f} bits "
      f"(prevalence {np.mean(real_labels):.4f})")
assert math.isclose(H_diabetes, 0.5825, abs_tol=1e-3)

The picture below shows entropy for every possible prevalence of a binary label. The real dataset sits on the left shoulder of the curve: a
problem that is *mostly predictable* (say "no" and you are right 86% of the time) and therefore one where a question has to be very good to
remove even a tenth of a bit. Remember this shape; it explains a lot of what happens in Part 4.

In [ ]:
p_grid = np.linspace(0.001, 0.999, 400)
H_grid = [entropy_from_probabilities([p, 1 - p]) for p in p_grid]
prev = float(np.mean(real_labels))

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(p_grid, H_grid, color=PALETTE["blue"])
ax.plot([prev], [H_diabetes], "o", color=PALETTE["orange"], markersize=8, markeredgecolor="white", markeredgewidth=2)
ax.annotate(f"BRFSS diabetes label\np = {prev:.3f}, H = {H_diabetes:.2f} bits", (prev, H_diabetes), xytext=(0.22, 0.45),
            textcoords="axes fraction", color="#52514e", arrowprops=dict(arrowstyle="-", color="#898781", lw=1))
style_axes(ax, title="Entropy of a yes/no label as a function of prevalence", xlabel="probability of the positive class", ylabel="entropy (bits)")
ax.set_ylim(0, 1.05)
plt.show()

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q01 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

In your own words: why does a group that is half diabetic and half not have an entropy of exactly 1 bit, and why does a group where everyone has the same label have 0 bits? What is a 'bit' measuring here? Finish by saying the entropy you computed for the real diabetes label column and what that number tells you about the problem.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q01")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q02 &nbsp;&middot;&nbsp; aim for 30&ndash;75 seconds</strong>

Describe what happened when you first ran entropy_from_probabilities on the pure group [1.0, 0.0]. Why does the formula try to compute log(0)? What is the limit of p times log2(p) as p goes to 0, and what exactly did you change in your code so that a pure group returns 0?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q02")

## Part 2 &mdash; Information gain: choosing the best question

### 2.1 Sixteen patients you can hold in your head
Before touching a quarter of a million rows we work with a toy table small enough to compute by hand. The 16 patients below are synthetic but
use the same vocabulary as the real data: blood pressure, BMI class, age band, physical activity, and the diabetes label. The format is the
one from lecture: a **list of records**, each record a list, the **target in the last column**.

In [ ]:
toy_attributes = ["HighBP", "BMI_class", "Age_group", "PhysActive"]
toy_data = [
    # HighBP, BMI_class,    Age_group, PhysActive, Diabetes
    ["no",  "normal",     "young",  "yes", "no"],
    ["no",  "normal",     "middle", "yes", "no"],
    ["yes", "normal",     "senior", "no",  "no"],
    ["no",  "normal",     "senior", "yes", "no"],
    ["yes", "normal",     "middle", "no",  "no"],
    ["no",  "overweight", "young",  "yes", "no"],
    ["no",  "overweight", "middle", "no",  "no"],
    ["no",  "overweight", "middle", "yes", "no"],
    ["yes", "overweight", "middle", "yes", "no"],
    ["yes", "overweight", "senior", "no",  "yes"],
    ["yes", "overweight", "senior", "yes", "yes"],
    ["yes", "obese",      "middle", "no",  "yes"],
    ["yes", "obese",      "senior", "no",  "yes"],
    ["no",  "obese",      "senior", "no",  "yes"],
    ["yes", "obese",      "young",  "no",  "yes"],
    ["no",  "obese",      "young",  "yes", "no"],
]


def get_labels(data: List[List]) -> List:
    """The target values of a dataset in list-of-records format (target in the last column)."""
    return [record[-1] for record in data]


pd.DataFrame(toy_data, columns=toy_attributes + ["Diabetes"], index=[f"P{i + 1:02d}" for i in range(len(toy_data))])

### 2.2 Information gain
Splitting the 16 patients on an attribute $A$ partitions them into one child group per value $v$. The expected entropy *after* asking about
$A$ is the size-weighted average of the children's entropies, often called the **remainder**; the **information gain** is how much uncertainty
the question removed:

$$\mathrm{Gain}(S, A) = H(S) - \sum_{v \in \mathrm{values}(A)} \frac{|S_v|}{|S|}\, H(S_v)$$

Children are weighted by size because a question that produces a tiny pure group and a huge mixed group has not helped much: most patients
are still in the mixed group. The best attribute is the one with the largest gain. That greedy choice, applied recursively, *is* ID3.

Two small functions get us there. `partition` is the workhorse you will reuse in Part 3.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>partition</code> and <code>information_gain</code></strong>

<code>partition(data, col)</code> returns a dictionary mapping each value found in column <code>col</code> to the list of records with that value (keep the records in their original order). <code>information_gain(data, col)</code> implements the formula above using <code>entropy</code>, <code>get_labels</code> and <code>partition</code>.
</div>

In [ ]:
def partition(data: List[List], col: int) -> Dict:
    """Group records by the value in column `col`: {value: [records with that value]}, preserving record order."""
    # YOUR CODE HERE
    raise NotImplementedError()


def information_gain(data: List[List], col: int) -> float:
    """Entropy of the whole dataset minus the size-weighted entropy of the groups produced by splitting on column `col`."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
groups = partition(toy_data, 1)
assert sorted(groups) == ["normal", "obese", "overweight"]
assert [len(groups[k]) for k in ("normal", "overweight", "obese")] == [5, 6, 5]
assert groups["normal"][0] == toy_data[0], "records keep their order"
assert math.isclose(information_gain(toy_data, 1), 0.3845, abs_tol=1e-3), "BMI_class"
assert math.isclose(information_gain(toy_data, 0), 0.2054, abs_tol=1e-3), "HighBP"
assert math.isclose(information_gain(toy_data, 2), 0.1635, abs_tol=1e-3), "Age_group"
assert math.isclose(information_gain(toy_data, 3), 0.2054, abs_tol=1e-3), "PhysActive"
print("partition / information_gain: checks passed")

A small helper to look at every attribute at once. You will use it on the real data too.

In [ ]:
def gain_table(data: List[List], attribute_names: List[str], score=None) -> pd.DataFrame:
    """Score every attribute of a dataset with `score(data, col)` (default information_gain), best first."""
    score = score or information_gain
    rows = [{"attribute": name, "distinct_values": len(partition(data, col)), "score": score(data, col)}
            for col, name in enumerate(attribute_names)]
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)


print(f"H(Diabetes) on the toy patients = {entropy(get_labels(toy_data)):.4f} bits")
gain_table(toy_data, toy_attributes)

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q03 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Walk me through the information gain of BMI_class on the toy patients: the entropy of the parent, the entropy of each of the three child groups, the weight each child gets, the remainder, and the final gain. Say the numbers you got. Then explain why each child's entropy is weighted by its size instead of being averaged equally.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q03")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #2: the hospital's export has one more column</strong>

Real exports carry identifiers. Suppose the toy table came with a <code>patient_id</code> column and you, reasonably, passed every column to the algorithm. Run the two cells below and look at which attribute wins.
</div>

In [ ]:
toy_attributes_with_id = ["patient_id"] + toy_attributes
toy_data_with_id = [[f"P{i + 1:02d}"] + record for i, record in enumerate(toy_data)]
gain_table(toy_data_with_id, toy_attributes_with_id)

### 2.3 Why information gain loves identifiers
`patient_id` wins with a gain of 0.954 bits, which is the entire entropy of the label: 16 branches, each holding one patient, each perfectly
pure. The algorithm is not wrong about the training data. It is wrong about the world: the 17th patient has an id the tree has never seen, so
the split tells us nothing. In general, **information gain is biased towards attributes with many values**, because many small groups are
easier to make pure, whether or not the attribute means anything. You will see the real-data version of this in Part 5, where a tree happily
splits BMI into one branch per integer value.

Quinlan's C4.5 answers with the **gain ratio**: divide the gain by the *split information*, which is the entropy of the partition itself
(how many bits it takes to say which branch a patient went down):

$$\mathrm{SplitInfo}(S, A) = -\sum_{v} \frac{|S_v|}{|S|} \log_2 \frac{|S_v|}{|S|}
\qquad
\mathrm{GainRatio}(S, A) = \frac{\mathrm{Gain}(S, A)}{\mathrm{SplitInfo}(S, A)}$$

A 16-way split costs $\log_2 16 = 4$ bits of split information; a 2-way split costs at most 1. Attributes that fragment the data pay for it.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>split_info</code> and <code>gain_ratio</code></strong>

Notice that <code>split_info</code> is just the entropy of the branch sizes: reuse <code>entropy_from_probabilities</code>. If the split information is 0 (every record has the same value, so the attribute cannot split anything) return a gain ratio of 0.
</div>

In [ ]:
def split_info(data: List[List], col: int) -> float:
    """Entropy of the partition induced by column `col`: how many bits it costs to say which branch a record went down."""
    # YOUR CODE HERE
    raise NotImplementedError()


def gain_ratio(data: List[List], col: int) -> float:
    """Information gain of column `col` divided by its split information (0 when the column has a single value)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert math.isclose(split_info(toy_data_with_id, 0), 4.0), "16 equal branches cost log2(16) = 4 bits"
assert math.isclose(split_info(toy_data, 0), 1.0), "an 8/8 split costs 1 bit"
assert math.isclose(gain_ratio(toy_data_with_id, 0), 0.2386, abs_tol=1e-3), "patient_id"
assert math.isclose(gain_ratio(toy_data, 1), 0.2434, abs_tol=1e-3), "BMI_class"
assert gain_ratio([["a", "x"], ["a", "y"]], 0) == 0.0, "an attribute with a single value has ratio 0"
print("split_info / gain_ratio: checks passed")

In [ ]:
gain_table(toy_data_with_id, toy_attributes_with_id, score=gain_ratio)

Gain ratio puts `BMI_class` back on top, but look at the margin: 0.243 against 0.239. The bias is reduced, not removed. C4.5 therefore adds a
guard (only consider attributes whose plain gain is at least average), and modern libraries sidestep the issue altogether by only ever
making **binary** splits, which you will implement in Part 5. The honest practical answer is the simplest: never feed an identifier to a
learning algorithm.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q04 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

When you added patient_id, information gain chose it as the best attribute. Explain why its gain equals the parent's entropy, and what would happen if the tree used that split on the next patient who walks in. What does gain ratio divide by, which attribute did it prefer in your run, and by how much? Does gain ratio fully fix the problem?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q04")

## Part 3 &mdash; Growing a tree: the ID3 algorithm

### 3.1 From one question to a tree
ID3 (Quinlan, 1986) is the greedy recursion you have probably guessed:

```
id3(patients, attributes):
    if every patient has the same label        -> leaf with that label
    if no attributes are left                  -> leaf with the majority label
    if the depth limit is reached              -> leaf with the majority label
    A = the attribute with the highest gain
    for each value v of A:
        child_v = id3(patients with A == v, attributes without A)
        (if no patients have A == v            -> leaf with the majority label of the PARENT)
    return a node that tests A and points to the children
```

Three details matter more than they look:

* **Majority at every node.** We store the majority label at *every* node, not only at leaves. A leaf uses it as its prediction; an internal
  node keeps it as a fallback. You will find out why in 3.4.
* **Ties.** When two attributes have the same gain, take the one that comes **first in `attribute_names`**. When two labels tie for the
  majority, take the one that **appears first in the data**. Fixed rules make your tree reproducible and gradable.
* **Attributes are used once per path.** After splitting on `BMI_class` nobody below that node asks about `BMI_class` again (its value is
  already known). This is why the recursion carries a list of *remaining candidate columns*.

### 3.2 The tree as a data structure
A `Node` holds what the recursion knows about a group of patients. Leaves have `feature is None`. Internal nodes map each value of the
split attribute to a child. The `threshold` field stays `None` for now; Part 5 uses it for numeric splits such as `BMI <= 28.5`.

In [ ]:
@dataclass
class Node:
    """One node of a decision tree. A leaf has feature None; its prediction is the majority class of its training records."""
    prediction: object                      # majority label of the records that reached this node
    n_samples: int                          # how many training records reached it
    class_counts: dict                      # label -> count, for those records
    feature: Optional[str] = None           # attribute tested here (None for a leaf)
    col: Optional[int] = None               # column index of that attribute
    threshold: Optional[float] = None       # numeric splits only (Part 5): left child if value <= threshold
    children: dict = field(default_factory=dict)   # value -> Node   (or "<=" / ">" -> Node for numeric splits)

    def is_leaf(self) -> bool:
        """True when the node makes a prediction instead of asking a question."""
        return self.feature is None


def print_tree(node: Node, indent: int = 0, max_depth: int = 4) -> None:
    """Text picture of a tree: one line per branch, leaves marked with ->, deeper subtrees summarised past max_depth."""
    pad = "    " * indent
    if node.is_leaf():
        print(f"{pad}-> {node.prediction}   n={node.n_samples} {node.class_counts}")
        return
    if indent >= max_depth:
        print(f"{pad}[{node.feature} ...] subtree with {count_leaves(node)} leaves (not shown)")
        return
    for value, child in node.children.items():
        test = f"{node.feature} {value} {node.threshold}" if node.threshold is not None else f"{node.feature} == {value}"
        print(f"{pad}[{test}]   n={child.n_samples}")
        print_tree(child, indent + 1, max_depth)


def count_leaves(node: Node) -> int:
    """Number of leaves below (and including) a node."""
    return 1 if node.is_leaf() else sum(count_leaves(c) for c in node.children.values())


def count_nodes(node: Node) -> int:
    """Total number of nodes in a tree."""
    return 1 + sum(count_nodes(c) for c in node.children.values())


def tree_depth(node: Node) -> int:
    """Length of the longest root-to-leaf path (a lone leaf has depth 0)."""
    return 0 if node.is_leaf() else 1 + max(tree_depth(c) for c in node.children.values())


def tree_to_dict(node: Node):
    """Convert a tree to the nested-dictionary format from lecture: {(feature, col, value): subtree-or-label}."""
    if node.is_leaf():
        return node.prediction
    return {(node.feature, node.col, value): tree_to_dict(child) for value, child in node.children.items()}

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>majority_class</code> and <code>best_attribute</code></strong>

<code>majority_class(labels)</code>: the most frequent label; on a tie, the label that appears first in the list (<code>Counter</code> remembers insertion order, and <code>max</code> returns the first maximum it meets). <code>best_attribute(data, candidates, score)</code>: among the candidate column indices, the one with the highest score; on a tie, the candidate listed first. <code>score</code> defaults to <code>information_gain</code> so you can swap in <code>gain_ratio</code> later.
</div>

In [ ]:
def majority_class(labels: List):
    """The most common label; ties go to the label that appears first in the list."""
    # YOUR CODE HERE
    raise NotImplementedError()


def best_attribute(data: List[List], candidates: List[int], score: Callable = None) -> int:
    """Column index (from `candidates`) with the highest score; ties go to the candidate that comes first."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert majority_class(["no", "no", "no", "yes"]) == "no"
assert majority_class(["no", "no", "yes", "yes"]) == "no", "tie -> first seen"
assert majority_class(["yes", "no", "yes", "no"]) == "yes", "tie -> first seen"
assert best_attribute(toy_data, [0, 1, 2, 3]) == 1, "BMI_class has the highest gain"
assert best_attribute(toy_data, [0, 3]) == 0, "HighBP and PhysActive tie -> first candidate"
assert best_attribute(toy_data, [3, 0]) == 3, "ties follow candidate order"
assert best_attribute(toy_data_with_id, [0, 1, 2, 3, 4], score=gain_ratio) == 2, "gain ratio prefers BMI_class (column 2 here)"
print("majority_class / best_attribute: checks passed")

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>id3</code></strong>

Follow the pseudocode in 3.1 and return <code>Node</code> objects. <code>candidates=None</code> means all columns are available. <code>depth_limit=None</code> means no limit; <code>depth_limit=0</code> returns a single leaf. Fill <code>n_samples</code> and <code>class_counts</code> (use <code>dict(Counter(labels))</code>) for every node; <code>print_tree</code> and Part 5 rely on them. Pass the current node's majority down as <code>parent_majority</code> so that an empty branch (<code>data == []</code>) becomes a leaf predicting it. The recursion arguments <code>depth</code> and <code>parent_majority</code> are for internal use.
</div>

In [ ]:
def id3(data: List[List], attribute_names: List[str], depth_limit: Optional[int] = None, candidates: Optional[List[int]] = None,
        depth: int = 0, parent_majority=None, score: Callable = None) -> Node:
    """Grow a decision tree with ID3 (greedy, recursive, categorical attributes) and return its root Node."""
    # YOUR CODE HERE
    raise NotImplementedError()

**Self-check against two trees you know.** The toy tree (verify it by hand: the root is `BMI_class`, the overweight branch splits on
`Age_group`, the obese branch on `PhysActive`), and the exact tree from lecture, in the nested-dictionary format from lecture.

In [ ]:
data_lecture = [['round', 'large', 'blue', 'no'], ['square', 'large', 'green', 'yes'], ['square', 'small', 'red', 'no'],
                ['round', 'large', 'red', 'yes'], ['square', 'small', 'blue', 'no'], ['round', 'small', 'blue', 'no'],
                ['round', 'small', 'red', 'yes'], ['square', 'small', 'green', 'no'], ['round', 'large', 'green', 'yes'],
                ['square', 'large', 'green', 'yes'], ['square', 'large', 'red', 'no'], ['square', 'large', 'green', 'yes'],
                ['round', 'large', 'red', 'yes'], ['square', 'small', 'red', 'no'], ['round', 'small', 'green', 'no']]
attribute_names_lecture = ['shape', 'size', 'color']
LECTURE_TREE = {('size', 1, 'large'): {('color', 2, 'blue'): 'no', ('color', 2, 'green'): 'yes',
                                       ('color', 2, 'red'): {('shape', 0, 'round'): 'yes', ('shape', 0, 'square'): 'no'}},
                ('size', 1, 'small'): {('shape', 0, 'square'): 'no',
                                       ('shape', 0, 'round'): {('color', 2, 'blue'): 'no', ('color', 2, 'red'): 'yes', ('color', 2, 'green'): 'no'}}}
TOY_TREE = {('BMI_class', 1, 'normal'): 'no',
            ('BMI_class', 1, 'overweight'): {('Age_group', 2, 'young'): 'no', ('Age_group', 2, 'middle'): 'no', ('Age_group', 2, 'senior'): 'yes'},
            ('BMI_class', 1, 'obese'): {('PhysActive', 3, 'no'): 'yes', ('PhysActive', 3, 'yes'): 'no'}}

toy_tree = id3(toy_data, toy_attributes)
assert tree_to_dict(toy_tree) == TOY_TREE, "the toy tree"
assert tree_to_dict(id3(data_lecture, attribute_names_lecture)) == LECTURE_TREE, "the lecture tree"
assert tree_to_dict(id3(data_lecture, attribute_names_lecture, depth_limit=1)) == {('size', 1, 'large'): 'yes', ('size', 1, 'small'): 'no'}
stump = id3(data_lecture, attribute_names_lecture, depth_limit=0)
assert stump.is_leaf() and stump.prediction == 'no' and stump.n_samples == 15 and stump.class_counts == {'no': 8, 'yes': 7}
assert toy_tree.children['overweight'].prediction == 'no' and toy_tree.children['overweight'].n_samples == 6, "internal nodes carry majority and counts"
assert id3([], toy_attributes, parent_majority='yes').prediction == 'yes', "an empty branch inherits the parent's majority"
print("id3: checks passed")

In [ ]:
print_tree(toy_tree)

Read the printout top to bottom: a normal-BMI patient is cleared immediately; an overweight patient is asked their age; an obese patient is
asked whether they exercise. Notice that `HighBP`, the attribute with the *second*-highest gain at the root, is never used. Keep that in
mind for Q07.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q05 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

List every condition under which your id3 function stops splitting and returns a leaf, and say what label that leaf gets in each case. In particular: what do you return for an empty branch, and why is the parent's majority the right choice?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q05")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q07 &nbsp;&middot;&nbsp; aim for 30&ndash;60 seconds</strong>

HighBP had the second-highest information gain at the root of the toy data, yet it does not appear anywhere in the final tree. Explain how a greedy, recursive algorithm can end up never using a globally useful feature.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q07")

### 3.3 Using the tree: prediction
Prediction is a walk from the root: at each internal node read the patient's value for the node's feature, step into that child, stop at
a leaf and return its prediction.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>predict_one_naive</code></strong>

The literal walk described above, nothing more. Keep this version: a later cell shows what is wrong with it, and the tests expect it to stay naive.
</div>

In [ ]:
def predict_one_naive(record: List, node: Node):
    """Walk the tree following the record's attribute values and return the leaf's prediction (literal version)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert [predict_one_naive(r, toy_tree) for r in toy_data] == get_labels(toy_data), "a full-depth tree reproduces its training labels"
lecture_tree = id3(data_lecture, attribute_names_lecture)
assert [predict_one_naive(r, lecture_tree) for r in data_lecture] == get_labels(data_lecture)
print("predict_one_naive: checks passed")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #3: a patient the tree has never seen</strong>

A new patient arrives: high blood pressure, <b>underweight</b>, young, not active. Nobody in the 16 training patients was underweight. Run the cell below and look carefully at what comes back, then at what happens when that answer is scored.
</div>

In [ ]:
new_patient = ["yes", "underweight", "young", "no"]
print("prediction for the underweight patient:", predict_one_naive(new_patient, toy_tree))


def error_rate(predictions: List, data: List[List]) -> float:
    """Fraction of records whose prediction differs from the true label; refuses missing predictions."""
    truth = get_labels(data)
    for p, record in zip(predictions, data):
        if p is None:
            raise ValueError(f"the tree produced no prediction for {record[:-1]}; every record must get a label")
    return sum(p != t for p, t in zip(predictions, truth)) / len(data)


scored = toy_data + [new_patient + ["no"]]
try:
    print("error rate:", error_rate([predict_one_naive(r, toy_tree) for r in scored], scored))
except ValueError as problem:
    print("scoring failed ->", problem)

### 3.4 Why the walk fell off the tree, and the fallback
The root asks `BMI_class`, looks for a child named `underweight`, finds none, and the walk has nowhere to go. In a notebook this returns
`None`; in production it is a crash in the middle of a clinic. It will happen constantly on real data: with 84 distinct BMI values, a test
fold will always contain values a training fold never saw.

The principled fallback is already sitting in the node: **when the walk cannot continue, predict the majority label of the node you are
standing on**. That node summarises every training patient who answered the earlier questions the same way as the new patient; it is the
most informed guess the tree can make. This is why ID3 stores `prediction` at internal nodes, and it is also how the "empty branch" case
of the pseudocode is justified: the same guess, made at training time instead of prediction time.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>predict_one</code> with the fallback</strong>

Same walk, but when the child for a value does not exist, return the current node's <code>prediction</code>. Then fill in <code>classify</code>, which maps the walk over a list of records.
</div>

In [ ]:
def predict_one(record: List, node: Node):
    """Walk the tree to a leaf; if a value has no branch, fall back to the majority class of the current node."""
    # YOUR CODE HERE
    raise NotImplementedError()


def classify(tree: Node, records: List[List]) -> List:
    """Predictions for every record in a list."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert predict_one_naive(new_patient, toy_tree) is None, "keep the naive version naive"
assert predict_one(new_patient, toy_tree) == "no", "root majority is 'no' (10 of 16)"
assert predict_one(["yes", "obese", "young", "unknown"], toy_tree) == "yes", "falls back at the obese node (4 of 5 are 'yes')"
assert classify(toy_tree, toy_data) == get_labels(toy_data)
assert error_rate(classify(lecture_tree, data_lecture), data_lecture) == 0.0
assert math.isclose(error_rate(classify(id3(data_lecture, attribute_names_lecture, depth_limit=0), data_lecture), data_lecture), 7 / 15), "the majority stump is wrong on the 7 'yes' of 15"
print("predict_one / classify: checks passed")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q06 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

Your naive predictor returned None for the underweight patient. Explain exactly where the tree walk got stuck and why. Describe your fix. Why is the majority class stored at that node a sensible fallback, and what would be a worse alternative?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q06")

### 3.5 Where you stand
You now own a complete, working decision-tree learner: entropy, gain, gain ratio, a greedy recursive builder with a depth limit, and a
predictor that never falls off the tree. Everything that follows is about *using it well* and *finding out where it breaks*.

## Part 4 &mdash; Real patients and honest metrics

### 4.1 From 16 toy patients to 253,680 real ones
Your `id3` works on lists of lists. The real data arrives as a table of integers, and for the experiments ahead we want two views of it:

* **arrays** (`numpy`): a matrix `X` with one row per respondent and one column per feature, and a vector `y` of 0/1 labels. Parts 5 to 7
  work on these; they make vectorised operations (and the thousands of trees in an ensemble) practical.
* **lists of records**, exactly like the toy data, so that the code you wrote in Part 3 can run on real patients right now.

We also split the data once and for all. A random **20% holdout** (`X_test, y_test`, 50,736 respondents) is locked away until a model
is evaluated; it is sampled *stratified*, so that it has the same 13.9% prevalence as the whole. From the remaining 80% we draw a stratified
**development sample of 50,000** respondents (`X_dev, y_dev`) for the many experiments in Parts 4 to 6; the full training set returns in Part 7.
Why not use all 203k rows for every experiment? Because you will train hundreds of trees, and a 4x smaller sample makes every experiment 4x faster
at no cost to the lessons. Finally, seven features are **numeric or ordered** (BMI, age band, days of poor health, ...); Part 5 treats them
differently from the yes/no ones, so we record their column indices in `NUMERIC_COLS`.

In [ ]:
def stratified_split(y: np.ndarray, test_fraction: float, seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """Index arrays (train, test) for a random split that keeps the class proportions in both parts."""
    rng = np.random.default_rng(seed)
    test = []
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c))
        test.extend(idx[: int(round(test_fraction * idx.size))].tolist())
    test = np.array(sorted(test))
    train_mask = np.ones(y.size, dtype=bool)
    train_mask[test] = False
    return np.flatnonzero(train_mask), test


def stratified_subsample(y: np.ndarray, n: int, seed: int = 0) -> np.ndarray:
    """Indices of a random subsample of size about n with the same class proportions as y."""
    rng = np.random.default_rng(seed)
    parts = [rng.choice(np.flatnonzero(y == c), size=int(round(n * np.mean(y == c))), replace=False) for c in np.unique(y)]
    return np.concatenate(parts)


feature_names = [c for c in diabetes_df.columns if c != "Diabetes_binary"]
X_all = diabetes_df[feature_names].to_numpy(dtype=np.int64)
y_all = diabetes_df["Diabetes_binary"].to_numpy(dtype=np.int64)
NUMERIC_FEATURES = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]
NUMERIC_COLS = [feature_names.index(f) for f in NUMERIC_FEATURES]

train_idx, test_idx = stratified_split(y_all, 0.20, seed=0)
X_train, y_train, X_test, y_test = X_all[train_idx], y_all[train_idx], X_all[test_idx], y_all[test_idx]
dev_idx = stratified_subsample(y_train, 50_000, seed=0)
X_dev, y_dev = X_train[dev_idx], y_train[dev_idx]

dev_rows = [list(map(int, row)) + [int(label)] for row, label in zip(X_dev, y_dev)]      # list-of-records view
test_rows = [list(map(int, row)) + [int(label)] for row, label in zip(X_test, y_test)]

print(f"train {X_train.shape}  holdout {X_test.shape}  dev {X_dev.shape}")
print(f"prevalence: all {y_all.mean():.4f}  train {y_train.mean():.4f}  holdout {y_test.mean():.4f}  dev {y_dev.mean():.4f}")
print("numeric / ordered columns:", NUMERIC_FEATURES)
assert X_dev.shape == (50000, 21) and X_test.shape == (50736, 21)

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #4: the tree that would not learn</strong>

Your <code>id3</code> from Part 3 is about to train on 50,000 real respondents at depth limits 0 to 6 and be scored on the 50,736-patient holdout. Depth 0 is the tree that asks no questions at all. Run the cell (about half a minute) and compare the rows. Before reading on, decide: did the tree learn anything? Is your code broken?
</div>

In [ ]:
def accuracy_of(tree: Node, rows: List[List]) -> float:
    """Fraction of records a tree labels correctly (list-of-records view)."""
    return 1.0 - error_rate(classify(tree, rows), rows)


sweep = []
for d in [0, 1, 2, 3, 4, 5, 6]:
    t0 = time.time()
    tree = id3(dev_rows, feature_names, depth_limit=d)
    sweep.append({"depth_limit": d, "root": tree.feature, "leaves": count_leaves(tree), "fit_seconds": round(time.time() - t0, 2),
                  "train_accuracy": accuracy_of(tree, dev_rows), "holdout_accuracy": accuracy_of(tree, test_rows)})
pd.DataFrame(sweep).round(4)

### 4.2 Why 86% is a number about the patients, not about the tree
Nothing is broken. Diabetes prevalence is 13.9%, so the depth-0 tree, which predicts "no diabetes" for everyone, is right 86.1% of the time.
A deeper tree only changes a prediction when some leaf contains **more diabetic than non-diabetic patients**, and at prevalence 14% that takes
several questions: the first splits produce children that are *less mixed* (the entropy goes down, your gain is real) but still majority
"no", so every leaf still says "no" and the accuracy is identical to the digit. When leaves finally flip, deeper trees get *worse* on the
holdout even as they get better on the training sample; Part 5 is about that.

The lesson is older than machine learning: **accuracy hides what happens to the minority class**, and in screening, the minority class is
the whole point. Clinicians use four numbers instead. With "has diabetes" as the positive class:

| | predicted positive | predicted negative |
|---|---|---|
| **actually positive** | true positive (TP) | false negative (FN): the patient we missed |
| **actually negative** | false positive (FP): the unnecessary blood test | true negative (TN) |

* **Sensitivity** (recall, true-positive rate) $= \dfrac{TP}{TP + FN}$: of the patients who have the disease, how many did we flag?
* **Specificity** (true-negative rate) $= \dfrac{TN}{TN + FP}$: of the healthy, how many did we correctly leave alone?
* **Precision** (positive predictive value) $= \dfrac{TP}{TP + FP}$: of those we flagged, how many really have it?
* **F1** $= \dfrac{2 \cdot \text{precision} \cdot \text{sensitivity}}{\text{precision} + \text{sensitivity}}$: one number that is only high when both are.
* **Balanced accuracy** $= \tfrac{1}{2}(\text{sensitivity} + \text{specificity})$: accuracy as if both classes were equally common.

A *screening* test wants high sensitivity (missing a case is costly; a false alarm costs a confirmatory test). A *confirmatory* test before a
treatment with side effects wants high specificity and precision. Keep both use-cases in mind for Q09.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement the confusion counts and four metrics</strong>

<code>confusion_counts(y_true, y_pred, positive=1)</code> returns the tuple <code>(tp, fp, fn, tn)</code>; accept lists or arrays (wrap them in <code>np.asarray</code>). The four metric functions take those four counts and must return 0.0 instead of dividing by zero (a tree that flags nobody has undefined precision; we call it 0).
</div>

In [ ]:
def confusion_counts(y_true, y_pred, positive=1) -> Tuple[int, int, int, int]:
    """(tp, fp, fn, tn) for a list/array of true labels and predictions, with `positive` as the positive class."""
    # YOUR CODE HERE
    raise NotImplementedError()


def sensitivity(tp: int, fp: int, fn: int, tn: int) -> float:
    """TP / (TP + FN): the fraction of true cases that were flagged (0.0 if there are no true cases)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def specificity(tp: int, fp: int, fn: int, tn: int) -> float:
    """TN / (TN + FP): the fraction of healthy patients correctly left alone (0.0 if there are none)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def precision(tp: int, fp: int, fn: int, tn: int) -> float:
    """TP / (TP + FP): the fraction of flagged patients who really have the condition (0.0 if nobody was flagged)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def f1_score(tp: int, fp: int, fn: int, tn: int) -> float:
    """Harmonic mean of precision and sensitivity (0.0 when both are 0)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def accuracy(tp: int, fp: int, fn: int, tn: int) -> float:
    """(TP + TN) / everything."""
    total = tp + fp + fn + tn
    return (tp + tn) / total if total else 0.0


def balanced_accuracy(tp: int, fp: int, fn: int, tn: int) -> float:
    """Mean of sensitivity and specificity."""
    return 0.5 * (sensitivity(tp, fp, fn, tn) + specificity(tp, fp, fn, tn))


METRICS = {"accuracy": accuracy, "sensitivity": sensitivity, "specificity": specificity,
           "precision": precision, "f1": f1_score, "balanced_acc": balanced_accuracy}


def evaluate(y_true, y_pred) -> Dict[str, float]:
    """All six metrics of a set of predictions, as a dictionary."""
    counts = confusion_counts(y_true, y_pred)
    return {name: fn(*counts) for name, fn in METRICS.items()}

In [ ]:
y_t = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
y_p = [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
assert confusion_counts(y_t, y_p) == (1, 1, 2, 6)
assert math.isclose(sensitivity(1, 1, 2, 6), 1 / 3) and math.isclose(specificity(1, 1, 2, 6), 6 / 7)
assert math.isclose(precision(1, 1, 2, 6), 0.5) and math.isclose(f1_score(1, 1, 2, 6), 0.4)
assert sensitivity(0, 0, 0, 10) == 0.0 and precision(0, 0, 3, 7) == 0.0 and f1_score(0, 0, 3, 7) == 0.0, "no division by zero"
assert confusion_counts(np.array([1, 0]), np.array([1, 0])) == (1, 0, 0, 1)
assert confusion_counts(["yes", "no"], ["no", "no"], positive="yes") == (0, 0, 1, 1), "works for string labels too"
m = evaluate(y_test, np.zeros_like(y_test))
assert math.isclose(m["accuracy"], 0.8607, abs_tol=1e-3) and m["sensitivity"] == 0.0 and m["specificity"] == 1.0
print("confusion_counts and metrics: checks passed")

Now the same depth sweep, scored properly. Read the sensitivity column.

In [ ]:
sweep = []
for d in [0, 1, 2, 3, 4, 5, 6]:
    tree = id3(dev_rows, feature_names, depth_limit=d)
    row = {"depth_limit": d, "leaves": count_leaves(tree)}
    row.update({f"holdout_{k}": v for k, v in evaluate(get_labels(test_rows), classify(tree, test_rows)).items()})
    sweep.append(row)
sweep_df = pd.DataFrame(sweep).set_index("depth_limit").round(3)
sweep_df

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q08 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Your trees from depth 0 upward all scored about 86% on the real patients, and deeper trees were not better. Explain why, using the prevalence of diabetes in the data and what a leaf needs to contain before its prediction changes. Say the sensitivity you measured at depth 0 and at the deepest tree. Is an 86%-accurate screening tool useful?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q08")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q09 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Define sensitivity and specificity in terms of the four cells of your confusion matrix, and state your numbers for one of your trees. If this tree were used as a first-pass diabetes screen in a clinic, which of the two matters more and why? Which would matter more for a confirmatory test before starting medication?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q09")

### 4.3 Honest evaluation
Three habits protect you from fooling yourself with numbers like the ones above.

1. **A holdout the model never saw.** Training accuracy climbs forever; only unseen patients tell you how the tool will do in the clinic.
2. **Stratification.** Random splits of a 14%-positive dataset can, by chance, hand a small test fold very few positives, making sensitivity a
   lottery. Keeping class proportions equal across folds removes that noise.
3. **Cross-validation.** One holdout gives one number; $k$ folds give $k$ numbers, and their spread tells you how much to trust the mean.
   The function you write below produces the folds; `cross_validate` (provided) runs any `fit` / `predict` pair over them and is used in Part 5.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>stratified_folds</code></strong>

Return a list of <code>k</code> index arrays that partition <code>range(len(y))</code> so that every fold has (almost) the same class proportions as <code>y</code>. One way: for each class, shuffle its indices with <code>np.random.default_rng(seed)</code>, cut them into <code>k</code> chunks with <code>np.array_split</code>, and give chunk <i>i</i> to fold <i>i</i>. Folds must be disjoint and cover every index.
</div>

In [ ]:
def stratified_folds(y: np.ndarray, k: int, seed: int = 0) -> List[np.ndarray]:
    """k disjoint index arrays covering all of y, each with (almost) the same class proportions as y."""
    # YOUR CODE HERE
    raise NotImplementedError()


def cross_validate(X: np.ndarray, y: np.ndarray, folds: List[np.ndarray], fit: Callable, predict: Callable) -> pd.DataFrame:
    """Train with fit(X_train, y_train) and score with predict(model, X) on every fold; one row of train/test metrics per fold."""
    rows = []
    for i, test_fold in enumerate(folds):
        train_fold = np.concatenate([f for j, f in enumerate(folds) if j != i])
        model = fit(X[train_fold], y[train_fold])
        row = {"fold": i}
        row.update({f"train_{k}": v for k, v in evaluate(y[train_fold], predict(model, X[train_fold])).items()})
        row.update({f"test_{k}": v for k, v in evaluate(y[test_fold], predict(model, X[test_fold])).items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index("fold")

In [ ]:
folds = stratified_folds(y_dev, 5, seed=0)
assert len(folds) == 5
assert sorted(np.concatenate(folds).tolist()) == list(range(len(y_dev))), "folds cover every index exactly once"
assert all(abs(len(f) - 10000) <= 2 for f in folds), "folds have (almost) equal sizes"
assert all(abs(y_dev[f].mean() - y_dev.mean()) < 0.005 for f in folds), "every fold keeps the prevalence"
assert len(set(map(len, stratified_folds(np.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1]), 2)))) <= 2
print("stratified_folds: checks passed")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#128270; A subtlety the data hides: duplicates</strong>

BRFSS answers are coarse (yes/no, 1&ndash;13 age bands, integer BMI), so two different people can produce identical rows. Run the cell below.
</div>

In [ ]:
dup_mask = diabetes_df.duplicated(keep=False)
print(f"rows that have at least one exact duplicate elsewhere in the table: {dup_mask.sum():,} ({dup_mask.mean():.1%})")
print(f"rows that are an exact copy of an earlier row: {diabetes_df.duplicated().sum():,}")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q10 &nbsp;&middot;&nbsp; aim for 30&ndash;75 seconds</strong>

About 14% of the rows in this dataset have an exact duplicate. If a patient's row appears in both the training fold and the test fold, what happens to the estimated test performance, and is that leakage? What would you do about it before trusting a cross-validation number?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q10")

## Part 5 &mdash; The limits of a single tree

### 5.1 Porting the tree to arrays
Three things your Part 3 tree cannot do yet, and that every serious implementation does:

1. **Numeric thresholds.** ID3 treats `BMI == 27` and `BMI == 28` as unrelated categories. A tree should be able to ask `BMI <= 28.5`.
2. **Pre-pruning.** A leaf built on 3 patients is noise; `min_samples_leaf` forbids such splits.
3. **Random feature subsets** at each node (`max_features`), which random forests in Part 6 need.

Doing this on lists of lists would be painfully slow once ensembles multiply the work by hundreds of trees, so from here on trees work on
`numpy` arrays: `X` is an integer matrix, `y` a vector of class codes `0..n_classes-1`. The algorithm is the one you already wrote; only the
bookkeeping changes. Start by porting the two functions everything else depends on.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Port <code>entropy</code> and <code>information_gain</code> to arrays</strong>

<code>entropy_np(y)</code>: counts via <code>np.bincount</code> (labels are small non-negative integers), probabilities, then the formula; drop zero counts to avoid log(0) (you know why), and return 0.0 for an empty array. <code>information_gain_np(X, y, col)</code>: loop over <code>np.unique(X[:, col])</code>, select each group's labels with a boolean mask, and apply the same weighted formula as before.
</div>

In [ ]:
def entropy_np(y: np.ndarray) -> float:
    """Entropy in bits of an integer label array (0.0 if empty)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def information_gain_np(X: np.ndarray, y: np.ndarray, col: int) -> float:
    """Information gain of splitting on every distinct value of column `col` (array version of information_gain)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def encode_table(data: List[List], attribute_names: List[str]) -> Tuple[np.ndarray, np.ndarray, List[Dict]]:
    """Turn a list-of-records table with string values into integer arrays X, y (codes assigned in sorted order per column)."""
    columns = list(zip(*data))
    codes = [{v: i for i, v in enumerate(sorted(set(column)))} for column in columns]
    matrix = np.array([[codes[j][v] for j, v in enumerate(record)] for record in data], dtype=np.int64)
    return matrix[:, :-1], matrix[:, -1], codes


def split_info_np(X: np.ndarray, col: int) -> float:
    """Entropy of the branch sizes produced by splitting on column `col`."""
    _, counts = np.unique(X[:, col], return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


def gain_ratio_np(X: np.ndarray, y: np.ndarray, col: int) -> float:
    """Information gain divided by split information (0 when the column has a single value)."""
    si = split_info_np(X, col)
    return 0.0 if si == 0 else information_gain_np(X, y, col) / si

In [ ]:
toy_X, toy_y, toy_codes = encode_table(toy_data, toy_attributes)
assert toy_codes[-1] == {"no": 0, "yes": 1}
assert math.isclose(entropy_np(toy_y), entropy(get_labels(toy_data)))
assert entropy_np(np.array([], dtype=np.int64)) == 0.0 and entropy_np(np.array([1, 1, 1])) == 0.0
assert math.isclose(entropy_np(np.array([0, 1, 2, 3])), 2.0)
assert math.isclose(entropy_np(y_dev), entropy(get_labels(dev_rows)))
for col in range(4):
    assert math.isclose(information_gain_np(toy_X, toy_y, col), information_gain(toy_data, col)), toy_attributes[col]
col_genhlth = feature_names.index("GenHlth")
assert math.isclose(information_gain_np(X_dev, y_dev, col_genhlth), information_gain(dev_rows, col_genhlth), abs_tol=1e-9)
assert math.isclose(gain_ratio_np(toy_X, toy_y, 1), gain_ratio(toy_data, 1))
t0 = time.time(); information_gain(dev_rows, col_genhlth); t_list = time.time() - t0
t0 = time.time(); information_gain_np(X_dev, y_dev, col_genhlth); t_np = time.time() - t0
print(f"entropy_np / information_gain_np: checks passed  (one gain on 50k rows: lists {t_list*1000:.0f} ms, arrays {t_np*1000:.1f} ms)")

### 5.2 Numeric thresholds
For an ordered feature the tree asks a yes/no question `x <= t`. Candidate thresholds are the midpoints between consecutive distinct values
(any $t$ between two neighbouring values gives the same split, so the midpoint is a canonical choice). For each candidate compute the gain
of the two-way split and keep the best. A threshold split can never produce 84 branches, which is how CART (Breiman et al., 1984) and every
modern library avoid the many-values bias you met in Part 2.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>best_threshold</code></strong>

Return <code>(threshold, gain)</code> for the best <code>x <= threshold</code> split of column <code>col</code>, or <code>(None, -inf)</code> if no split is possible. Skip candidates that would leave fewer than <code>min_samples_leaf</code> rows on either side. Ties go to the smallest threshold (use a strict <code>&gt;</code> when comparing gains).
</div>

In [ ]:
def best_threshold(X: np.ndarray, y: np.ndarray, col: int, min_samples_leaf: int = 1) -> Tuple[Optional[float], float]:
    """Best midpoint threshold t for the split x <= t on column `col` and its information gain; (None, -inf) if none is valid."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
X_small = np.array([[1], [2], [3], [4], [5], [6]])
y_small = np.array([0, 0, 0, 1, 1, 1])
t, g = best_threshold(X_small, y_small, 0)
assert t == 3.5 and math.isclose(g, 1.0), "a perfect split between 3 and 4"
assert best_threshold(X_small, y_small, 0, min_samples_leaf=4) == (None, -np.inf), "no split leaves 4 rows on both sides"
assert best_threshold(np.array([[7], [7], [7]]), np.array([0, 1, 0]), 0)[0] is None, "a constant column cannot be split"
col_bmi = feature_names.index("BMI")
t_bmi, g_bmi = best_threshold(X_dev, y_dev, col_bmi)
# an independent check: the gain of the chosen threshold equals the gain of the binarised column, and no other midpoint beats it
binarised = (X_dev[:, col_bmi] <= t_bmi).astype(np.int64).reshape(-1, 1)
assert math.isclose(g_bmi, information_gain_np(binarised, y_dev, 0))
vals = np.unique(X_dev[:, col_bmi])
others = [information_gain_np((X_dev[:, col_bmi] <= (a + b) / 2).astype(np.int64).reshape(-1, 1), y_dev, 0) for a, b in zip(vals[:-1], vals[1:])]
assert g_bmi >= max(others) - 1e-12
print(f"best_threshold: checks passed  (dev data: BMI <= {t_bmi} with gain {g_bmi:.4f} bits)")

### 5.3 The general tree
The recursion below is `id3` again with three additions, split into small functions so that each is testable. Read the provided pieces,
then write the three that carry the logic.

* `TreeSpec` bundles the hyper-parameters so that they do not have to be threaded through every recursive call.
* `make_node` builds a `Node` from a label array (majority by `argmax`, so the lower class code wins ties; an empty array becomes a leaf
  with the parent's majority).
* `choose_pool` returns the candidate columns to consider at this node: all of them, or a random subset of size `max_features`.
* `split_score` scores one column: for a numeric column, the best threshold; for a categorical column, gain or gain ratio of the multi-way
  split; it returns `(None, None)` when the column cannot be split under `min_samples_leaf`.
* **`should_stop`** (you): the stopping rule.
* **`best_split`** (you): the best column (and threshold) among the candidates.
* **`grow_tree`** (you): the recursion.
* `train_tree` is the public entry point; `predict_tree` predicts a whole matrix at once, walking the index sets down the tree, and reports
  which rows hit an unseen categorical value (the fallback from Part 3 applies).

In [ ]:
@dataclass
class TreeSpec:
    """Hyper-parameters shared by every recursive call of grow_tree."""
    feature_names: list
    numeric_cols: frozenset = frozenset()
    depth_limit: Optional[int] = None
    min_samples_leaf: int = 1
    max_features: Optional[int] = None
    criterion: str = "gain"                       # "gain" or "gain_ratio"
    rng: Optional[np.random.Generator] = None
    n_classes: int = 2


def make_node(y: np.ndarray, spec: TreeSpec, parent_majority) -> Node:
    """A leaf-shaped Node summarising a label array (an empty array inherits the parent's majority)."""
    if y.size == 0:
        return Node(prediction=parent_majority, n_samples=0, class_counts={})
    counts = np.bincount(y, minlength=spec.n_classes)
    return Node(prediction=int(counts.argmax()), n_samples=int(y.size), class_counts={c: int(k) for c, k in enumerate(counts) if k})


def choose_pool(candidates: List[int], spec: TreeSpec) -> List[int]:
    """The candidate columns examined at a node: all of them, or a random subset of size max_features (random forests)."""
    if spec.max_features is None or spec.max_features >= len(candidates):
        return candidates
    return sorted(spec.rng.choice(candidates, size=spec.max_features, replace=False).tolist())


def binary_entropy(p: float) -> float:
    """Entropy of a two-way split with proportions p and 1-p."""
    return 0.0 if p <= 0 or p >= 1 else float(-(p * math.log2(p) + (1 - p) * math.log2(1 - p)))


def split_score(X: np.ndarray, y: np.ndarray, col: int, spec: TreeSpec) -> Tuple[Optional[float], Optional[float]]:
    """(threshold, score) for splitting on `col` under the spec's criterion; (None, None) if the column cannot be split."""
    if col in spec.numeric_cols:
        t, gain = best_threshold(X, y, col, spec.min_samples_leaf)
        if t is None:
            return None, None
        return t, gain / binary_entropy(float((X[:, col] <= t).mean())) if spec.criterion == "gain_ratio" else gain
    values, counts = np.unique(X[:, col], return_counts=True)
    if len(values) < 2 or counts.min() < spec.min_samples_leaf:
        return None, None
    return None, gain_ratio_np(X, y, col) if spec.criterion == "gain_ratio" else information_gain_np(X, y, col)

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>should_stop</code>, <code>best_split</code> and <code>grow_tree</code></strong>

<b>should_stop</b>: stop when the node is pure, when no candidates remain, when <code>depth &gt;= spec.depth_limit</code> (if a limit is set), or when the node has fewer than <code>2 * spec.min_samples_leaf</code> samples (no split could satisfy the minimum). <b>best_split</b>: call <code>split_score</code> on every candidate and return <code>(col, threshold, score)</code> of the best; skip <code>(None, None)</code> results; ties go to the earlier candidate; return <code>(None, None, -inf)</code> if nothing can be split. <b>grow_tree</b>: make the node; stop if told to; pick the best split from <code>choose_pool(candidates, spec)</code>; if there is none, return the leaf; otherwise set <code>feature</code>, <code>col</code>, <code>threshold</code> and recurse: a categorical split creates one child per value (key = the integer value, candidates minus that column), a numeric split creates children <code>"&lt;="</code> and <code>"&gt;"</code> (numeric columns stay available below). Pass <code>node.prediction</code> down as the children's <code>parent_majority</code>.
</div>

In [ ]:
def should_stop(node: Node, candidates: List[int], depth: int, spec: TreeSpec) -> bool:
    """True when a node must become a leaf: pure, out of candidates, at the depth limit, or too small to split."""
    # YOUR CODE HERE
    raise NotImplementedError()


def best_split(X: np.ndarray, y: np.ndarray, candidates: List[int], spec: TreeSpec) -> Tuple[Optional[int], Optional[float], float]:
    """(col, threshold, score) of the best split among the candidate columns; (None, None, -inf) if none is possible."""
    # YOUR CODE HERE
    raise NotImplementedError()


def grow_tree(X: np.ndarray, y: np.ndarray, candidates: List[int], depth: int, parent_majority, spec: TreeSpec) -> Node:
    """Recursive builder: returns the subtree for the rows (X, y) using the candidate columns."""
    # YOUR CODE HERE
    raise NotImplementedError()


def train_tree(X: np.ndarray, y: np.ndarray, feature_names: List[str], numeric_cols=(), depth_limit: Optional[int] = None,
               min_samples_leaf: int = 1, max_features: Optional[int] = None, criterion: str = "gain",
               rng: Optional[np.random.Generator] = None) -> Node:
    """Train a decision tree on integer arrays. Columns in numeric_cols get threshold splits; the others get one branch per value."""
    spec = TreeSpec(list(feature_names), frozenset(numeric_cols), depth_limit, min_samples_leaf, max_features, criterion,
                    rng if rng is not None else np.random.default_rng(0), int(y.max()) + 1)
    return grow_tree(X, y, list(range(X.shape[1])), 0, None, spec)


def _predict_into(node: Node, X: np.ndarray, idx: np.ndarray, out: np.ndarray, unseen: np.ndarray) -> None:
    """Fill out[idx] by routing the rows idx through the subtree rooted at node."""
    if node.is_leaf() or idx.size == 0:
        out[idx] = node.prediction
        return
    x = X[idx, node.col]
    if node.threshold is not None:
        left = x <= node.threshold
        _predict_into(node.children["<="], X, idx[left], out, unseen)
        _predict_into(node.children[">"], X, idx[~left], out, unseen)
        return
    seen = np.zeros(idx.size, dtype=bool)
    for value, child in node.children.items():
        mask = x == value
        seen |= mask
        _predict_into(child, X, idx[mask], out, unseen)
    out[idx[~seen]] = node.prediction          # the Part 3 fallback: majority of the node where the walk got stuck
    unseen[idx[~seen]] = True


def predict_tree(node: Node, X: np.ndarray, return_unseen: bool = False):
    """Predicted class codes for every row of X (optionally also a mask of rows that met an unseen categorical value)."""
    out = np.empty(len(X), dtype=np.int64)
    unseen = np.zeros(len(X), dtype=bool)
    _predict_into(node, X, np.arange(len(X)), out, unseen)
    return (out, unseen) if return_unseen else out

In [ ]:
# 1. the array tree reproduces your Part 3 tree on the toy data and the lecture data
tree_np = train_tree(toy_X, toy_y, toy_attributes)
assert count_leaves(tree_np) == count_leaves(toy_tree) == 6 and tree_np.feature == "BMI_class"
assert predict_tree(tree_np, toy_X).tolist() == toy_y.tolist()
lec_X, lec_y, lec_codes = encode_table(data_lecture, attribute_names_lecture)
lec_tree_np = train_tree(lec_X, lec_y, attribute_names_lecture)
assert count_leaves(lec_tree_np) == count_leaves(lecture_tree) and predict_tree(lec_tree_np, lec_X).tolist() == lec_y.tolist()
assert train_tree(lec_X, lec_y, attribute_names_lecture, depth_limit=0).is_leaf()
# 2. an unseen categorical value falls back to the node majority and is reported
pred, unseen = predict_tree(tree_np, np.array([[1, 9, 0, 0]]), return_unseen=True)
assert pred.tolist() == [0] and unseen.tolist() == [True]
# 3. real data: the first question with threshold splits is HighBP, then general health on both sides
real2 = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS, depth_limit=2)
assert real2.feature == "HighBP" and real2.threshold is None, real2.feature
assert all(child.feature == "GenHlth" and child.threshold == 2.5 for child in real2.children.values())
assert set(real2.children) == {0, 1} and set(real2.children[0].children) == {"<=", ">"}
# 4. min_samples_leaf is respected everywhere
def leaf_sizes(node):
    """Sample counts of every leaf below a node."""
    return [node.n_samples] if node.is_leaf() else [s for c in node.children.values() for s in leaf_sizes(c)]
pruned = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=500)
assert min(leaf_sizes(pruned)) >= 500 and count_leaves(pruned) < 200
# 5. the categorical tree on dev data is your ID3: same root and same holdout accuracy as Part 4's depth-2 tree
cat2 = train_tree(X_dev, y_dev, feature_names, depth_limit=2)
assert cat2.feature == "GenHlth" and math.isclose(evaluate(y_test, predict_tree(cat2, X_test))["accuracy"], sweep_df.loc[2, "holdout_accuracy"], abs_tol=1e-3)
# 6. max_features draws a subset
assert len(choose_pool(list(range(21)), TreeSpec(feature_names, max_features=4, rng=np.random.default_rng(1)))) == 4
print("should_stop / best_split / grow_tree: checks passed")

In [ ]:
print_tree(real2)

Three questions, eight leaves, and already a readable screening rule: high blood pressure, then self-rated health, then BMI. Keep this
small tree in mind when the big ones below start to behave badly.

### 5.4 Experiment: how deep should a tree be?
Train trees of increasing depth on the development sample and score them on the sample they were trained on and on the holdout.
The plot on the left is the most important picture in this project.

In [ ]:
curve_rows = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None]:
    t0 = time.time()
    tree = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS, depth_limit=d, min_samples_leaf=5)
    tr, te = evaluate(y_dev, predict_tree(tree, X_dev)), evaluate(y_test, predict_tree(tree, X_test))
    curve_rows.append({"depth_limit": "none" if d is None else d, "actual_depth": tree_depth(tree), "leaves": count_leaves(tree),
                       "train_error": 1 - tr["accuracy"], "holdout_error": 1 - te["accuracy"],
                       "holdout_sensitivity": te["sensitivity"], "holdout_f1": te["f1"], "seconds": time.time() - t0})
curve = pd.DataFrame(curve_rows)
curve.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.9))
x = curve["actual_depth"]
ax1.plot(x, curve["train_error"], "o-", color=PALETTE["blue"], label="training sample", markeredgecolor="white", markeredgewidth=1.5)
ax1.plot(x, curve["holdout_error"], "o-", color=PALETTE["orange"], label="holdout", markeredgecolor="white", markeredgewidth=1.5)
ax1.axhline(1 - y_test.mean(), color=PALETTE["gray"], lw=1, ls=":")
ax1.text(x.max(), 1 - y_test.mean() + 0.003, "predict 'no' for everyone", ha="right", va="bottom", fontsize=8, color="#52514e")
ax1.legend(loc="center left")
style_axes(ax1, title="Error rate vs tree depth (min_samples_leaf = 5)", xlabel="depth of the trained tree", ylabel="error rate")
ax2.plot(x, curve["holdout_sensitivity"], "o-", color=PALETTE["blue"], label="sensitivity", markeredgecolor="white", markeredgewidth=1.5)
ax2.plot(x, curve["holdout_f1"], "o-", color=PALETTE["aqua"], label="F1", markeredgecolor="white", markeredgewidth=1.5)
ax2.legend(loc="upper left")
style_axes(ax2, title="What the holdout minority class sees", xlabel="depth of the trained tree", ylabel="score on the holdout")
plt.tight_layout()
plt.show()

Left: training error falls monotonically (a deep enough tree memorises its sample), while holdout error reaches a minimum around a
depth of 5 to 8 and then climbs well *above* the predict-nothing line. The gap between the curves is **overfitting**: the tree is learning
the noise of 50,000 particular people rather than the population. Right: the deep, overfitted trees do flag more diabetics (sensitivity rises)
at the price of many false alarms; the two charts together are why a single tree needs both a complexity control and a decision rule chosen
with the clinical use in mind.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q11 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Describe the shape of your training-error and test-error curves as the depth limit grows. At what depth did test error reach its minimum in your run? Why does training error keep falling while test error turns around, and what is the name for what happens on the right side of the plot?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q11")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #5: 84 kinds of BMI</strong>

Train your categorical ID3 (every column as categories, exactly what Part 3 did) and the threshold tree on the same 50,000 patients, both with no depth limit, and compare them on the holdout. Then look at what the categorical tree did with BMI.
</div>

In [ ]:
cat_tree = train_tree(X_dev, y_dev, feature_names)                                   # Part 3 semantics: every column categorical
thr_tree = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS)        # numeric columns get thresholds
comparison = []
for name, tree in [("categorical ID3", cat_tree), ("threshold splits", thr_tree)]:
    pred, unseen = predict_tree(tree, X_test, return_unseen=True)
    m = evaluate(y_test, pred)
    comparison.append({"tree": name, "leaves": count_leaves(tree), "depth": tree_depth(tree),
                       "train_accuracy": evaluate(y_dev, predict_tree(tree, X_dev))["accuracy"], "holdout_accuracy": m["accuracy"],
                       "holdout_sensitivity": m["sensitivity"], "holdout_f1": m["f1"], "unseen_branch_rate": unseen.mean()})
display(pd.DataFrame(comparison).set_index("tree").round(4))
print(f"root of the categorical tree: {cat_tree.feature} with {len(cat_tree.children)} branches")
for value, child in cat_tree.children.items():
    print(f"   {cat_tree.feature} == {value}: n={child.n_samples:,} -> next question {child.feature} with {len(child.children)} branches")

### 5.5 Fragmentation
Under `GenHlth == 4` and `GenHlth == 5`, information gain picked BMI and split the patients into one branch *per integer BMI value*: 68 and 58
branches, many holding one or two people, each a "pure" leaf by accident. That is Part 2's identifier problem in disguise: a many-valued
attribute buys training purity by **fragmenting** the data into groups too small to say anything about the population. The symptoms on
the holdout are all in the table: about twice as many leaves, lower sensitivity and F1 than the threshold tree (accuracy is similar, and
both full-depth trees sit far below the depth-limited ones), and about 8% of holdout patients landing on a branch that did not exist in
training (your Part 3 fallback silently handled every one of them). The threshold tree
asks `BMI <= 30.5` instead and can revisit BMI deeper down with a different cut, which is more statistical power per question.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q12 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

Compare the number of leaves and the unseen-branch rate of your categorical ID3 tree against the threshold tree, quoting your numbers. Why does splitting BMI into dozens of branches (one per integer value) hurt generalisation even though it raises training accuracy? How does a threshold split avoid the problem?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q12")

### 5.6 Pre-pruning with `min_samples_leaf`
The cheapest complexity control: refuse splits that would create a leaf with fewer than $m$ patients. Sweep $m$ and watch the holdout.
(Post-pruning methods such as reduced-error or cost-complexity pruning grow the full tree and cut back; scikit-learn's `ccp_alpha` is the
latter. Pre-pruning is enough for our purposes.)

In [ ]:
msl_rows = []
for m in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]:
    tree = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=m)
    te = evaluate(y_test, predict_tree(tree, X_test))
    msl_rows.append({"min_samples_leaf": m, "leaves": count_leaves(tree), "depth": tree_depth(tree),
                     "holdout_accuracy": te["accuracy"], "holdout_sensitivity": te["sensitivity"], "holdout_f1": te["f1"], "holdout_balanced_acc": te["balanced_acc"]})
msl_df = pd.DataFrame(msl_rows)
fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.plot(msl_df["min_samples_leaf"], msl_df["holdout_accuracy"], "o-", color=PALETTE["blue"], label="accuracy", markeredgecolor="white", markeredgewidth=1.5)
ax.plot(msl_df["min_samples_leaf"], msl_df["holdout_f1"], "o-", color=PALETTE["orange"], label="F1", markeredgecolor="white", markeredgewidth=1.5)
ax.plot(msl_df["min_samples_leaf"], msl_df["holdout_balanced_acc"], "o-", color=PALETTE["aqua"], label="balanced accuracy", markeredgecolor="white", markeredgewidth=1.5)
ax.set_xscale("log")
ax.legend(loc="center right")
style_axes(ax, title="Holdout scores of a full-depth tree vs min_samples_leaf", xlabel="min_samples_leaf (log scale)", ylabel="score")
plt.show()
msl_df.round(4)

Accuracy peaks around $m \approx 100$, F1 and balanced accuracy peak much earlier. There is no single best tree: the right complexity
depends on which mistakes you care about, which is the clinical question you will answer in Part 7.

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #6: two halves, two different stories</strong>

Split the development sample into two random halves and train the same tree (no depth limit, <code>min_samples_leaf = 5</code>) on each. If a tree were a stable description of the population, the two trees should agree. Run the cell and compare the roots, then the predictions.
</div>

In [ ]:
half_a, half_b = np.array_split(np.random.default_rng(1).permutation(len(y_dev)), 2)
tree_a = train_tree(X_dev[half_a], y_dev[half_a], feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=5)
tree_b = train_tree(X_dev[half_b], y_dev[half_b], feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=5)
pred_a, pred_b = predict_tree(tree_a, X_test), predict_tree(tree_b, X_test)
print(f"tree A: root {tree_a.feature} {'<= ' + str(tree_a.threshold) if tree_a.threshold is not None else ''}  leaves={count_leaves(tree_a)}  "
      f"holdout accuracy={evaluate(y_test, pred_a)['accuracy']:.4f}  F1={evaluate(y_test, pred_a)['f1']:.3f}")
print(f"tree B: root {tree_b.feature} {'<= ' + str(tree_b.threshold) if tree_b.threshold is not None else ''}  leaves={count_leaves(tree_b)}  "
      f"holdout accuracy={evaluate(y_test, pred_b)['accuracy']:.4f}  F1={evaluate(y_test, pred_b)['f1']:.3f}")
disagreement = float((pred_a != pred_b).mean())
print(f"the two trees give DIFFERENT predictions for {disagreement:.1%} of the holdout patients")
print("\n-- tree A, top two levels"); print_tree(tree_a, max_depth=2)
print("\n-- tree B, top two levels"); print_tree(tree_b, max_depth=2)

### 5.7 Instability, or why trees are high-variance models
Two halves of the same survey, two trees with similar accuracy, and one in seven patients gets a different answer depending on which half
was used. The greedy choice at the root is decided by small differences in gain; once it differs, everything below it differs, and a
full-depth tree keeps splitting until it has memorised the noise of its particular sample. In statistical terms a single tree has **high
variance**: its predictions depend heavily on the sample it happened to see. Part 6 turns this weakness into the idea behind random forests.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q13 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Trees A and B were trained on two halves of the same data. Compare their root splits and state the disagreement rate between their predictions on the test set. Why are decision trees high-variance models? Why should a physician be uneasy about a tool whose rules change this much when it is retrained?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q13")

### 5.8 What the tree reads from a patient
A tree is one of the few models a clinician can audit line by line, and that is both its strength and a responsibility. Count which features
the full-depth threshold tree actually uses.

In [ ]:
def split_counts(node: Node, counter: Optional[Counter] = None) -> Counter:
    """How many internal nodes test each feature."""
    counter = Counter() if counter is None else counter
    if not node.is_leaf():
        counter[node.feature] += 1
        for child in node.children.values():
            split_counts(child, counter)
    return counter


usage = split_counts(thr_tree)
print(f"the full-depth threshold tree has {sum(usage.values()):,} internal nodes; features tested:")
for feature, n in usage.most_common():
    print(f"   {feature:22s} {n:5d} splits")

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q14 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

Your tree uses Income and Education to split patients. Give one argument for allowing a clinical screening tool to use such features and one argument against. Say which side you land on and what you would want to check before deploying it.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q14")

### 5.9 Aside: trees draw staircases
Every split is a question about one feature, so a tree's decision boundary is made of axis-aligned pieces. A boundary that runs diagonally
through two features has to be approximated by a staircase, which costs depth and data. Two synthetic risk scores make the point.

In [ ]:
rng_demo = np.random.default_rng(3)
demo_X = rng_demo.integers(0, 40, size=(1500, 2))
demo_y = ((demo_X[:, 0] + demo_X[:, 1] + rng_demo.normal(0, 3, size=1500)) > 40).astype(np.int64)
demo_tree = train_tree(demo_X, demo_y, ["score A", "score B"], numeric_cols=[0, 1], depth_limit=6, min_samples_leaf=10)
gx, gy = np.meshgrid(np.arange(0, 40), np.arange(0, 40))
grid = np.column_stack([gx.ravel(), gy.ravel()])
region = predict_tree(demo_tree, grid).reshape(gx.shape)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.contourf(gx, gy, region, levels=[-0.5, 0.5, 1.5], colors=["#cde2fb", "#fbd9cb"], alpha=0.9)
ax.scatter(demo_X[demo_y == 0, 0], demo_X[demo_y == 0, 1], s=8, color=PALETTE["blue"], alpha=0.5, label="negative")
ax.scatter(demo_X[demo_y == 1, 0], demo_X[demo_y == 1, 1], s=8, color=PALETTE["orange"], alpha=0.5, label="positive")
ax.plot([0, 40], [40, 0], color="#0b0b0b", lw=1, ls="--", label="true boundary A + B = 40")
ax.legend(loc="upper right", fontsize=8)
style_axes(ax, title=f"A depth-{tree_depth(demo_tree)} tree approximating a diagonal", xlabel="score A", ylabel="score B")
ax.grid(False)
plt.show()

## Part 6 &mdash; Ensembles: many weak trees, one strong model

### 6.1 Averaging away the variance: bagging
Part 5 ended with a diagnosis: a single deep tree is accurate on its sample and unstable across samples. Statistics has a standard remedy for
a noisy estimator: **average many of them**. If you had 25 independent surveys you could grow 25 trees and let them vote; the idiosyncrasies
of each sample would cancel. You have one survey, so Breiman (1996) proposed to fake the 25 surveys with the **bootstrap**: draw $n$ rows
*with replacement* from the $n$ you have. Each bootstrap sample contains about 63% of the distinct rows (some twice or more) and misses the
rest, which is enough variation to grow visibly different trees. **Bootstrap aggregating** (bagging) is then:

1. for $b = 1 \dots B$: draw a bootstrap sample, grow an unpruned tree on it;
2. to predict, let the $B$ trees vote; the fraction voting "diabetes" is a usable probability.

Averaging reduces variance and leaves bias where it is: if every tree is systematically wrong in the same way, so is their average.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement bagging</strong>

<code>bootstrap_indices(n, rng)</code>: <code>n</code> indices drawn with replacement (<code>rng.integers</code>). <code>fit_bagging(X, y, n_trees, rng, **tree_kwargs)</code>: a list of trees, each trained with <code>train_tree(X[idx], y[idx], rng=rng, **tree_kwargs)</code> on its own bootstrap sample. <code>predict_proba_bagging(trees, X)</code>: the fraction of trees predicting class 1 for each row. <code>predict_bagging(trees, X, threshold=0.5)</code>: 1 where that fraction is at least the threshold, else 0.
</div>

In [ ]:
def bootstrap_indices(n: int, rng: np.random.Generator) -> np.ndarray:
    """n row indices drawn uniformly with replacement from range(n)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def fit_bagging(X: np.ndarray, y: np.ndarray, n_trees: int, rng: np.random.Generator, **tree_kwargs) -> List[Node]:
    """Train n_trees trees, each on its own bootstrap sample of (X, y); tree_kwargs are passed to train_tree."""
    # YOUR CODE HERE
    raise NotImplementedError()


def predict_proba_bagging(trees: List[Node], X: np.ndarray) -> np.ndarray:
    """Fraction of trees that vote for class 1, for each row of X."""
    # YOUR CODE HERE
    raise NotImplementedError()


def predict_bagging(trees: List[Node], X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Class 1 where the vote fraction reaches the threshold, class 0 elsewhere."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
idx = bootstrap_indices(10_000, np.random.default_rng(0))
assert idx.shape == (10_000,) and idx.min() >= 0 and idx.max() < 10_000
assert abs(len(np.unique(idx)) / 10_000 - 0.632) < 0.02, "a bootstrap sample holds about 63.2% distinct rows"
small_forest = fit_bagging(toy_X, toy_y, 7, np.random.default_rng(0), feature_names=toy_attributes)
assert len(small_forest) == 7 and all(isinstance(t, Node) for t in small_forest)
proba = predict_proba_bagging(small_forest, toy_X)
assert proba.shape == (16,) and proba.min() >= 0 and proba.max() <= 1 and np.allclose(proba * 7, np.round(proba * 7))
assert set(predict_bagging(small_forest, toy_X).tolist()) <= {0, 1}
assert np.array_equal(predict_bagging(small_forest, toy_X, threshold=0.0), np.ones(16, dtype=np.int64))
print("bagging: checks passed")

Now bag 25 full-depth trees (`min_samples_leaf = 5`) on the development sample. This is the longest cell in the project (one to two minutes):
you are growing 25 trees on 50,000 rows each in pure numpy. Watch what happens to the metrics as the number of voting trees grows, and
then what happens when you move the vote threshold.

In [ ]:
rng = np.random.default_rng(0)
t0 = time.time()
bagged = fit_bagging(X_dev, y_dev, 25, rng, feature_names=feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=5)
print(f"trained {len(bagged)} trees in {time.time() - t0:.0f} s")

single_tree = train_tree(X_dev, y_dev, feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=5)
rows = [{"model": "single tree", **evaluate(y_test, predict_tree(single_tree, X_test))}]
for B in [1, 5, 10, 25]:
    rows.append({"model": f"bagging, {B} trees", **evaluate(y_test, predict_bagging(bagged[:B], X_test))})
display(pd.DataFrame(rows).set_index("model").round(4))

root_counts = Counter(t.feature for t in bagged)
print("root feature of the 25 bagged trees:", dict(root_counts))

proba_bag = predict_proba_bagging(bagged, X_test)
thr_rows = [{"vote threshold": thr, **evaluate(y_test, (proba_bag >= thr).astype(int))} for thr in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1]]
display(pd.DataFrame(thr_rows).set_index("vote threshold").round(4))

In [ ]:
thresholds = np.linspace(0.02, 0.8, 40)
curves = pd.DataFrame([evaluate(y_test, (proba_bag >= t).astype(int)) for t in thresholds], index=thresholds)
fig, ax = plt.subplots(figsize=(6.8, 3.6))
for name, color in [("sensitivity", PALETTE["blue"]), ("specificity", PALETTE["orange"]), ("precision", PALETTE["aqua"])]:
    ax.plot(curves.index, curves[name], color=color, label=name)
ax.axvline(0.5, color=PALETTE["gray"], lw=1, ls=":")
ax.text(0.505, 0.02, "default vote threshold", fontsize=8, color="#52514e")
ax.legend(loc="center right")
style_axes(ax, title="Bagging: the vote threshold is a clinical dial", xlabel="fraction of trees voting 'diabetes' needed to flag a patient", ylabel="score on the holdout")
plt.show()

### 6.2 What averaging did, and what it did not
Two things happened at once. Accuracy went **up** with more trees, as promised: the averaged trees are less noisy. But at the default
0.5 vote threshold sensitivity went **down**: a patient is only flagged when a majority of trees agree, and for a 14%-prevalence condition
most borderline patients get a minority of votes. The vote fraction, however, is a *ranking*; lowering the threshold to 0.2 flags far more
true cases (sensitivity around 0.6) at a cost in false alarms, something a single tree, which outputs a hard label per leaf, could not offer.
Ensembles give you probabilities, and probabilities give you a dial.

And one thing did not happen: the trees are not very different from each other. Look at the root features: most of the 25 trees ask the
same first question. Bootstrap samples share 63% of their rows, and the strongest feature wins the root in nearly every one of them, so the
trees are **correlated** and the average of correlated estimators has more variance than the average of independent ones.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q15 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

Explain bootstrap aggregation in your own words. In your run, what did averaging 25 trees do to accuracy, sensitivity and F1 compared with the single tree at the default 0.5 vote threshold? What happened when you lowered the threshold, and why does the ensemble's vote fraction give you that knob when a single tree did not?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q15")

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #7: 25 trees that all ask the same question</strong>

Breiman's fix (2001) is the <b>random forest</b>: at every node, consider only a random subset of <code>max_features</code> columns. Trees are forced to find second-best questions, which decorrelates them. Your <code>grow_tree</code> already supports this through <code>choose_pool</code>. Train 25 trees with <code>max_features = 4</code> and compare roots, metrics and the ranking quality (AUC, provided).
</div>

In [ ]:
def auc_score(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Area under the ROC curve: the probability that a random positive is scored above a random negative (ties count half)."""
    y_true, scores = np.asarray(y_true), np.asarray(scores)
    uniq, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)
    average_rank = np.cumsum(counts) - (counts - 1) / 2.0
    ranks = average_rank[inverse]
    n_pos = int(y_true.sum()); n_neg = y_true.size - n_pos
    return float((ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


assert math.isclose(auc_score(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8])), 0.75)
assert math.isclose(auc_score(np.array([0, 1, 0, 1]), np.array([0.5, 0.5, 0.5, 0.5])), 0.5), "no information -> 0.5"

rng = np.random.default_rng(0)
t0 = time.time()
forest = fit_bagging(X_dev, y_dev, 25, rng, feature_names=feature_names, numeric_cols=NUMERIC_COLS, min_samples_leaf=5, max_features=4)
print(f"trained the random forest in {time.time() - t0:.0f} s")
print("root feature of the 25 forest trees:", dict(Counter(t.feature for t in forest)))
proba_rf = predict_proba_bagging(forest, X_test)

rows = []
for name, proba in [("bagging (25 trees)", proba_bag), ("random forest (25 trees, max_features=4)", proba_rf)]:
    for thr in [0.5, 0.2]:
        rows.append({"model": name, "vote threshold": thr, "AUC": auc_score(y_test, proba), **evaluate(y_test, (proba >= thr).astype(int))})
pd.DataFrame(rows).set_index(["model", "vote threshold"]).round(4)

The roots are now spread over a handful of features, and the forest's ranking quality (AUC) and its scores at a sensible threshold edge
above bagging. The gain on this dataset is modest, because two or three features carry most of the signal, but the mechanism is general and
it is why `RandomForestClassifier` is usually the first model anyone tries on tabular data.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q16 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

Count how many of your 25 bagged trees shared the same root feature. Why are bootstrapped trees so correlated, and why does correlation limit how much averaging can help? What does max_features change at each node, and how did the root features and the metrics change in your random-forest run?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q16")

### 6.3 Boosting: trees that learn from each other's mistakes
Bagging grows trees independently. **Boosting** grows them in sequence, each one concentrating on the patients the previous ones got wrong.
AdaBoost (Freund &amp; Schapire, 1997) keeps a weight $w_i$ on every training row (initially $1/n$) and repeats for $m = 1 \dots M$:

1. train a weak learner $h_m$ on the weighted data; here, a **decision stump** (a depth-1 tree);
2. compute its weighted error $\varepsilon_m = \sum_{i:\, h_m(x_i) \ne y_i} w_i$ and its vote $\alpha_m = \tfrac{1}{2}\ln\frac{1-\varepsilon_m}{\varepsilon_m}$;
3. update the weights: multiply by $e^{+\alpha_m}$ for misclassified rows and $e^{-\alpha_m}$ for correct ones, then renormalise;
4. stop early if $\varepsilon_m \ge 0.5$ (the learner is no better than chance on the weighted data).

The final prediction is the sign of $\sum_m \alpha_m\, h_m(x)$ with labels coded as $\pm 1$. Our `train_tree` does not take row weights,
so we use the classic **resampling** variant: draw a bootstrap sample *according to the weights* and train the stump on it; rows with large
weights appear many times. Boosting attacks bias rather than variance: stumps are terribly simple models, but each round is told exactly
where the previous ones failed.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>fit_adaboost</code></strong>

Follow the four steps. Code labels as +1/-1 with <code>np.where(y == 1, 1.0, -1.0)</code>. Draw the weighted sample with <code>rng.choice(n, size=n, replace=True, p=w)</code> and train the stump with <code>train_tree(X[sample], y[sample], depth_limit=1, rng=rng, **stump_kwargs)</code>. Compute the weighted error on <b>all</b> rows (not just the sample). Break out of the loop if the error is 0 or at least 0.5. Return three lists: stumps, alphas, errors.
</div>

In [ ]:
def fit_adaboost(X: np.ndarray, y: np.ndarray, n_rounds: int, rng: np.random.Generator, **stump_kwargs) -> Tuple[List[Node], List[float], List[float]]:
    """AdaBoost with decision stumps trained on weight-proportional resamples; returns (stumps, alphas, weighted errors)."""
    # YOUR CODE HERE
    raise NotImplementedError()


def adaboost_scores(stumps: List[Node], alphas: List[float], X: np.ndarray) -> np.ndarray:
    """Weighted vote sum_m alpha_m * h_m(x) with h in {-1, +1}; positive means class 1."""
    return sum(a * np.where(predict_tree(s, X) == 1, 1.0, -1.0) for s, a in zip(stumps, alphas))


def predict_adaboost(stumps: List[Node], alphas: List[float], X: np.ndarray, cutoff: float = 0.0) -> np.ndarray:
    """Class 1 where the weighted vote exceeds the cutoff (0 is the classic decision rule)."""
    return (adaboost_scores(stumps, alphas, X) > cutoff).astype(np.int64)

In [ ]:
rng = np.random.default_rng(0)
t0 = time.time()
stumps, alphas, errors = fit_adaboost(X_dev, y_dev, 50, rng, feature_names=feature_names, numeric_cols=NUMERIC_COLS)
print(f"{len(stumps)} rounds in {time.time() - t0:.0f} s")
assert len(stumps) == len(alphas) == len(errors) >= 20
assert all(tree_depth(s) <= 1 for s in stumps), "weak learners are stumps"
assert math.isclose(errors[0], y_dev.mean(), abs_tol=0.01), "with uniform weights the first stump predicts 'no' for all, so its error is the prevalence"
assert math.isclose(alphas[0], 0.5 * math.log((1 - errors[0]) / errors[0]))
assert all(a > 0 for a in alphas) and all(0 < e < 0.5 for e in errors)
assert errors[0] < errors[1] < 0.5, "the second round is harder: the diabetic rows now carry more weight"
assert set(predict_adaboost(stumps, alphas, X_test).tolist()) <= {0, 1}
print("fit_adaboost: checks passed")

In [ ]:
rows = [{"rounds": M, **evaluate(y_test, predict_adaboost(stumps[:M], alphas[:M], X_test)), "AUC": auc_score(y_test, adaboost_scores(stumps[:M], alphas[:M], X_test))}
        for M in [1, 2, 5, 10, 25, len(stumps)]]
display(pd.DataFrame(rows).set_index("rounds").round(4))
print("first eight stumps:", [(s.feature, s.threshold) for s in stumps[:8]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6))
ax1.plot(range(1, len(errors) + 1), errors, "o-", color=PALETTE["blue"], markeredgecolor="white", markeredgewidth=1.5)
ax1.axhline(0.5, color=PALETTE["gray"], lw=1, ls=":")
style_axes(ax1, title="Weighted error of each stump on the reweighted data", xlabel="boosting round", ylabel="weighted error")
ax2.plot(range(1, len(alphas) + 1), alphas, "o-", color=PALETTE["orange"], markeredgecolor="white", markeredgewidth=1.5)
style_axes(ax2, title="Vote weight (alpha) of each stump", xlabel="boosting round", ylabel="alpha")
plt.tight_layout()
plt.show()

### 6.4 Reading the boosting run
Round 1 is the depth-0 story again: with uniform weights the best stump still predicts "no" for everyone, its error is the prevalence (0.14)
and it earns a large $\alpha$. But every diabetic patient was misclassified, so their weights grow by $e^{\alpha} \approx 2.5$ while the
others shrink; by round 2 the reweighted data is far more balanced and the stump has to find a feature that separates diabetics. Error creeps
towards 0.5 and $\alpha$ towards 0 as the easy structure is used up. Fifty stumps, each a single question, end up matching a tuned deep tree,
and the weighted vote is again a score you can threshold.

| | Bagging / random forests | Boosting |
|---|---|---|
| trees are grown | independently, in parallel | sequentially, each correcting the last |
| each tree is | deep, low bias, high variance | shallow, high bias, low variance |
| the ensemble reduces | variance | bias (and some variance) |
| main risk | little: more trees never hurt | overfitting if run too long or with noisy labels |
| modern descendant | `RandomForestClassifier` | gradient boosting (`HistGradientBoostingClassifier`, XGBoost, LightGBM) |

Gradient boosting, the workhorse of applied tabular machine learning, generalises AdaBoost by fitting each new tree to the gradient of a
loss function; you meet it in Part 7.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q17 &nbsp;&middot;&nbsp; aim for 60&ndash;120 seconds</strong>

In AdaBoost, what happens to the weight of a misclassified patient after each round, and to a correctly classified one? The first stump's weighted error was about 0.14 in your run; explain why that gives a large alpha. Explain how the reweighting makes boosting pay attention to the diabetic minority that a single tree ignored.

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q17")

## Part 7 &mdash; Industry tools and the clinical decision

### 7.1 scikit-learn's trees
You have now built everything scikit-learn's `DecisionTreeClassifier` does, with three differences worth knowing: it implements CART
(binary splits only, even for categories, which must be numerically encoded), its default impurity is the **Gini index**
$1 - \sum_i p_i^2$ rather than entropy (`criterion="entropy"` switches), and its core loops are compiled, so the full 203k-row training set
takes well under a second. From here on use the **full training set** (`X_train, y_train`) and keep scoring on the untouched holdout with the
`evaluate` function you wrote; add scikit-learn's `roc_auc_score` for ranking quality.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Train a library tree and compare it with yours</strong>

Fit a <code>DecisionTreeClassifier</code> with <code>criterion="entropy"</code>, <code>max_depth=8</code> and <code>random_state=0</code> on <code>X_train, y_train</code>, store it in <code>sk_tree</code>, and put its holdout metrics in <code>sk_tree_metrics</code> (use <code>evaluate</code>). The provided cells then draw its top levels and compare it with your own depth-8 threshold tree trained on the same data.
</div>

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# YOUR CODE HERE
raise NotImplementedError()
print({k: round(v, 4) for k, v in sk_tree_metrics.items()}, "AUC", round(roc_auc_score(y_test, sk_tree.predict_proba(X_test)[:, 1]), 4))

In [ ]:
assert isinstance(sk_tree, DecisionTreeClassifier) and sk_tree.get_depth() == 8 and sk_tree.criterion == "entropy"
assert sk_tree.n_features_in_ == 21 and sk_tree.tree_.n_node_samples[0] == len(y_train), "trained on the full training set"
assert math.isclose(sk_tree_metrics["accuracy"], evaluate(y_test, sk_tree.predict(X_test))["accuracy"])
assert sk_tree_metrics["accuracy"] > 0.86 and sk_tree_metrics["sensitivity"] > 0.10
print("sk_tree: checks passed")

In [ ]:
t0 = time.time()
own_tree_8 = train_tree(X_train, y_train, feature_names, numeric_cols=NUMERIC_COLS, depth_limit=8)
own_seconds = time.time() - t0
t0 = time.time(); DecisionTreeClassifier(criterion="entropy", max_depth=8, random_state=0).fit(X_train, y_train); sk_seconds = time.time() - t0
pd.DataFrame([{"model": "your train_tree, depth 8", "fit_seconds": own_seconds, **evaluate(y_test, predict_tree(own_tree_8, X_test))},
              {"model": "sklearn DecisionTreeClassifier, depth 8", "fit_seconds": sk_seconds, **sk_tree_metrics}]).set_index("model").round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_tree(sk_tree, max_depth=2, feature_names=feature_names, class_names=["no diabetes", "diabetes"], proportion=True, impurity=True,
          rounded=True, fontsize=8, ax=ax)
ax.set_title("Top three levels of the scikit-learn tree (entropy, max_depth=8). 'value' = class proportions in the node.", loc="left", fontsize=10)
plt.show()

The same first questions you found by hand (`HighBP`, `GenHlth`, `BMI`, `Age`), the same numbers, in a tenth of a millisecond per split.
The point of building it yourself was never speed.

### 7.2 Forests and boosting at full scale

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Fit the three industry models</strong>

Fill the <code>models</code> dictionary with fitted estimators under <b>exactly</b> these keys: <code>"decision_tree"</code> (your <code>sk_tree</code>), <code>"random_forest"</code> (<code>RandomForestClassifier</code> with <code>n_estimators=100</code>, <code>min_samples_leaf=20</code>, <code>n_jobs=-1</code>, <code>random_state=0</code>) and <code>"gradient_boosting"</code> (<code>HistGradientBoostingClassifier</code> with <code>random_state=0</code>). If <code>xgboost</code> is installed, also add <code>"xgboost"</code> (<code>XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=0)</code>); wrap the import in <code>try/except ImportError</code> so the notebook runs without it. All on <code>X_train, y_train</code>.
</div>

In [ ]:
models = {}
# YOUR CODE HERE
raise NotImplementedError()

holdout_scores = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}
model_table = pd.DataFrame([{"model": name, "AUC": roc_auc_score(y_test, holdout_scores[name]), **evaluate(y_test, m.predict(X_test))}
                            for name, m in models.items()]).set_index("model")
model_table.round(4)

In [ ]:
assert {"decision_tree", "random_forest", "gradient_boosting"} <= set(models), "use the required keys"
assert isinstance(models["random_forest"], RandomForestClassifier) and models["random_forest"].n_estimators == 100
assert isinstance(models["gradient_boosting"], HistGradientBoostingClassifier)
assert model_table.loc["random_forest", "AUC"] > 0.80 and model_table.loc["gradient_boosting", "AUC"] > 0.81
assert model_table.loc["gradient_boosting", "AUC"] >= model_table.loc["decision_tree", "AUC"]
print("models: checks passed")

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.6))
for (name, scores), color in zip(holdout_scores.items(), SERIES):
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, color=color, label=f"{name} (AUC {roc_auc_score(y_test, scores):.3f})")
ax.plot([0, 1], [0, 1], color=PALETTE["gray"], lw=1, ls=":")
ax.legend(loc="lower right", fontsize=8)
style_axes(ax, title="ROC curves on the holdout", xlabel="1 - specificity (false-positive rate)", ylabel="sensitivity")
plt.show()

Every model has an accuracy near 86% and a sensitivity near 15% at the default decision rule, and the boosted models have the best ranking
quality (AUC around 0.83). Accuracy still cannot tell them apart; AUC and the operating point can.

### 7.3 Fixing the imbalance, and the trap inside the fix
A common reaction to "sensitivity 15%" is to **rebalance the training data**: keep every diabetic respondent and sample an equal number of
non-diabetics, train on that 50/50 set, and watch sensitivity jump. (The same Kaggle dataset ships a pre-balanced file; we build our own
from the training split instead, because that file was drawn from the same rows as our holdout and would leak.)

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>undersample_balanced</code></strong>

Return a shuffled index array containing <b>all</b> rows with <code>y == 1</code> and an equally large random sample of rows with <code>y == 0</code>, drawn without replacement using <code>rng</code>.
</div>

In [ ]:
def undersample_balanced(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Indices of a 50/50 subset: every positive row plus an equal number of randomly chosen negative rows."""
    # YOUR CODE HERE
    raise NotImplementedError()

<div style="background:#fdf0ea; border-left:6px solid #eb6834; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#129512; Deliberate problem #8: the model that was great until it met real prevalence</strong>

Train gradient boosting on the balanced set and score it twice: on a <i>balanced</i> holdout (the kind of test set a balanced-data tutorial would use) and on the <i>real</i> holdout with 13.9% prevalence. Compare the precision columns.
</div>

In [ ]:
rng = np.random.default_rng(0)
bal_idx = undersample_balanced(y_train, rng)
assert y_train[bal_idx].mean() == 0.5 and len(bal_idx) == 2 * y_train.sum() and len(np.unique(bal_idx)) == len(bal_idx)

balanced_model = HistGradientBoostingClassifier(random_state=0).fit(X_train[bal_idx], y_train[bal_idx])
bal_test_idx = undersample_balanced(y_test, rng)
rows = [{"evaluated on": "balanced holdout (50% prevalence)", **evaluate(y_test[bal_test_idx], balanced_model.predict(X_test[bal_test_idx]))},
        {"evaluated on": "real holdout (13.9% prevalence)", **evaluate(y_test, balanced_model.predict(X_test))}]
display(pd.DataFrame(rows).set_index("evaluated on").round(3))
tp, fp, fn, tn = confusion_counts(y_test, balanced_model.predict(X_test))
print(f"per 1,000 screened patients at real prevalence: {1000 * tp / len(y_test):.0f} true positives, {1000 * fp / len(y_test):.0f} false positives, "
      f"{1000 * fn / len(y_test):.0f} missed cases, {1000 * tn / len(y_test):.0f} correctly cleared")

### 7.4 Base rates
Sensitivity and specificity are properties of the *test*; precision is a property of the test **and the population**. Roughly: at 50%
prevalence, 79% sensitivity and 71% specificity give you about 79 true positives for every 29 false positives, so three flagged patients
in four really have diabetes. At 14% prevalence the same test produces about 110 true positives per 1,000 screened, but the 860 healthy
people now generate about 250 false positives, so fewer than one flagged patient in three is a true case. Nothing about the model changed;
the denominator did. Balanced training is a legitimate way to shift the operating point, but a balanced *test set* lies to you about
precision, and any deployment has to be judged at the prevalence it will meet.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q18 &nbsp;&middot;&nbsp; aim for 45&ndash;90 seconds</strong>

You trained on a balanced 50/50 sample and got sensitivity near 0.8, but precision collapsed when you evaluated at the true 14% prevalence, even though precision looked fine on a balanced test set. Explain this with base rates: out of 1,000 screened patients, roughly how many true positives and how many false positives does your model produce?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q18")

### 7.5 Choosing the operating point
The cleaner way to get the sensitivity a screening programme needs is to keep the model and move its **decision threshold**: every model
above outputs a probability, and a threshold of 0.5 is a convention, not a law. A screening programme might require 80% sensitivity; the
threshold is then whatever value delivers it on validation data, and specificity and precision are whatever follows.

<div style="background:#eaf7f1; border-left:6px solid #1baf7a; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#9997;&#65039; Implement <code>threshold_for_sensitivity</code></strong>

Given scores (higher = more likely diabetic) and true labels, return the <b>largest</b> threshold <i>t</i> such that predicting positive for <code>score &gt;= t</code> has sensitivity at least <code>target</code>. One approach: sort scores descending, compute the cumulative fraction of positives captured, and find the first position where it reaches the target.
</div>

In [ ]:
def threshold_for_sensitivity(scores: np.ndarray, y_true: np.ndarray, target: float = 0.80) -> float:
    """The largest threshold at which flagging score >= threshold reaches the target sensitivity."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
s = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2])
t = np.array([1, 0, 1, 1, 0, 1, 0, 0])
assert threshold_for_sensitivity(s, t, 0.75) == 0.6, "three of the four positives are captured at 0.6"
assert threshold_for_sensitivity(s, t, 1.0) == 0.4
for name, scores in holdout_scores.items():
    thr = threshold_for_sensitivity(scores, y_test, 0.80)
    m = evaluate(y_test, (scores >= thr).astype(int))
    assert m["sensitivity"] >= 0.80, name
    assert evaluate(y_test, (scores > thr).astype(int))["sensitivity"] < 0.80 or np.isclose(thr, scores.min()), f"{name}: not the largest threshold"
print("threshold_for_sensitivity: checks passed")

In [ ]:
operating = []
for name, scores in holdout_scores.items():
    thr = threshold_for_sensitivity(scores, y_test, 0.80)
    operating.append({"model": name, "threshold for 80% sensitivity": thr, **evaluate(y_test, (scores >= thr).astype(int))})
display(pd.DataFrame(operating).set_index("model").round(3))

gb_scores = holdout_scores["gradient_boosting"]
grid = np.linspace(0.02, 0.6, 59)
op = pd.DataFrame([evaluate(y_test, (gb_scores >= t).astype(int)) for t in grid], index=grid)
fig, ax = plt.subplots(figsize=(6.8, 3.6))
for name, color in [("sensitivity", PALETTE["blue"]), ("specificity", PALETTE["orange"]), ("precision", PALETTE["aqua"])]:
    ax.plot(op.index, op[name], color=color, label=name)
thr80 = threshold_for_sensitivity(gb_scores, y_test, 0.80)
ax.axvline(thr80, color=PALETTE["gray"], lw=1, ls=":")
ax.text(thr80 + 0.005, 0.03, f"80% sensitivity at {thr80:.3f}", fontsize=8, color="#52514e")
ax.legend(loc="center right")
style_axes(ax, title="Gradient boosting: scores as a function of the decision threshold", xlabel="probability threshold for flagging a patient", ylabel="score on the holdout")
plt.show()

### 7.6 What the forest thinks matters
Tree ensembles come with a free, if imperfect, explanation: how much each feature reduced impurity across all splits of all trees. Treat it
as "what the model used", not "what causes diabetes": these are self-reported survey answers, several are consequences rather than causes,
and correlated features share credit unpredictably.

In [ ]:
importances = pd.Series(models["random_forest"].feature_importances_, index=feature_names).sort_values()
fig, ax = plt.subplots(figsize=(6.4, 5.2))
ax.barh(importances.index, importances.values, color=PALETTE["blue"], height=0.6)
style_axes(ax, title="Random forest: mean impurity decrease per feature", xlabel="share of total impurity reduction")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=1); ax.grid(False, axis="y")
plt.show()

### 7.7 The capstone
You have the full picture a clinical team would need: half a dozen models, their ranking quality, their behaviour at the operating point a
screening programme would demand, the base-rate arithmetic, the features they rely on, and the failure modes you discovered on the way
(overfitting, fragmentation, instability, unseen values, leakage from duplicates, the accuracy illusion, the balanced-test illusion). Make
the recommendation.

<div style="background:#eef0fb; border-left:6px solid #6250d6; padding:12px 18px; margin:12px 0; border-radius:6px; color:#111;">
<strong>&#127897;&#65039; Voice question Q19 &nbsp;&middot;&nbsp; aim for 120&ndash;180 seconds</strong>

Capstone. You are presenting to a Hopkins clinical team deciding whether to deploy a diabetes screening model built from these survey indicators. Which model and operating threshold would you recommend, quoting its sensitivity, specificity and precision from your results? Describe two ways it could fail in practice, one ethical or practical concern, and what you would monitor after deployment. Finally, in one sentence, how has your understanding of decision trees changed since your mic-check answer?

Run the cell below, click <b>Record</b>, answer out loud, then click <b>Stop &amp; save</b>. You can re-record as often as you like; only the latest take is kept.
</div>

In [ ]:
record_answer("Q19")

### 7.8 Where this leads
* Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning* 1, 81&ndash;106. (ID3.)
* Quinlan, J. R. (1993). *C4.5: Programs for Machine Learning*. (Gain ratio, pruning, continuous attributes.)
* Breiman, L., Friedman, J., Olshen, R., Stone, C. (1984). *Classification and Regression Trees*. (CART, Gini, binary splits, cost-complexity pruning.)
* Breiman, L. (1996). Bagging predictors. *Machine Learning* 24, 123&ndash;140.
* Breiman, L. (2001). Random forests. *Machine Learning* 45, 5&ndash;32.
* Freund, Y., Schapire, R. E. (1997). A decision-theoretic generalization of on-line learning and an application to boosting. *JCSS* 55, 119&ndash;139. (AdaBoost.)
* Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *Annals of Statistics* 29, 1189&ndash;1232.
* Chen, T., Guestrin, C. (2016). XGBoost: a scalable tree boosting system. *KDD*.
* Hastie, Tibshirani, Friedman. *The Elements of Statistical Learning*, chapters 9, 10 and 15 (free online).

## Part 8 &mdash; Submit

1. Save the notebook, then **Kernel &rarr; Restart Kernel and Run All Cells**. Every cell must run without error, in order ([1], [2], ...).
   Your previous recordings are shown again by the recorder cells; nothing is lost by re-running.
2. Run the three cells below. The first lists the functions you wrote with docstring and length checks; fix anything flagged.
   The second lists your recordings. The third verifies everything and writes `submission.zip`.
3. **Save the notebook again** (the zip contains the file as saved on disk), re-run `build_submission()` if you changed anything,
   and upload `submission.zip` to Gradescope. Do not upload anything else.

In [ ]:
lint_notebook_functions()

In [ ]:
recording_status()

In [ ]:
build_submission()

<div style="color:#52514e; font-size:0.9em;">Dataset: CDC Behavioral Risk Factor Surveillance System 2015, cleaned by Alex Teboul and published on Kaggle under CC0.
Course infrastructure (recorder, checker) and this assignment: Johns Hopkins University, EN.601.464 Artificial Intelligence
(instructor [Professor's name]; teaching assistants Kaleb Aklilu and [Second TA's name]).
Released under the MIT License; source and contribution guide at https://github.com/kalk-ak/artificial_intelligence_decision_tree_hw.</div>